
# PRE-02 — UNSW-NB15 v3.5: Preprocessing & Representation

**Experiment 1 v2 — accepted source + fingerprint-multiplicity partition**

This notebook refits preprocessing **from scratch** on the accepted PRE-01 v3.4 membership.

### v3.5 lineage correction
PRE-02 no longer requires the superseded corrected-v2 PRE-01 directory to remain present on Drive. The accepted v3.4 partition is bound **directly to PRE-00** by an exact ordered `(row_id, feature_fingerprint)` identity digest across the full dataset, in addition to PRE-00's own cryptographic readiness checks.

No source-only PRE-02 fitted object, median, scaler, category vocabulary, transformed matrix, representation hash, or corruption manifest is reused.


## Scientific stage boundary and prohibited operations

This stage is deterministic preprocessing/representation construction only.

- Learned preprocessing is fitted on `benign_weight_train` only.
- The five frozen v3 roles may be mechanically transformed.
- Final-test outcomes remain evaluation-sealed.
- No AE/DAE model is trained here.
- No threshold is fitted here.
- No attack metric or model-selection result is computed here.
- Successful PRE-02 completion authorizes **one benign-only smoke AE**, not formal Notebook 01.


In [1]:
# ============================================================
# 3. EDITABLE CONFIGURATION — SET PROSPECTIVE VALUES BEFORE FITTING
# ============================================================
from pathlib import Path

DATASET_NAME = 'UNSW-NB15'
DATASET_PREFIX = 'UNSWNB15'
DATASET_FOLDER = 'UNSW-NB15'
PROTOCOL_VERSION = "Experiment_1_v2"
NOTEBOOK_STAGE = "PRE-02_SOURCE_MULTIPLICITY_V3"
PROJECT_FOLDER_NAME = "EXPERIMENT1_V2"

# -------------------------
# Authoritative path inputs
# -------------------------
# Leave a path as None to use conservative exact-filename discovery under the
# verified project/dataset root. Ambiguity is a hard failure; the notebook never guesses.
PROJECT_ROOT = None
MASTER_DATASET_PATH = None

GLOBAL_PROTOCOL_PATH = None
MASTER_MANIFEST_PATH = None
TAXONOMY_MANIFEST_PATH = None
ROW_IDENTITY_MANIFEST_PATH = None
RAW_FEATURE_POLICY_PATH = None
FINGERPRINT_AUDIT_PATH = None
FINGERPRINT_ROW_MAPPING_PATH = None

PRE00_READINESS_REPORT_PATH = None

PARTITION_POLICY_MANIFEST_PATH = None
PARTITION_MEMBERSHIP_PATH = None
PRE01_INTEGRITY_REPORT_PATH = None
PREPROCESSING_HANDOFF_MANIFEST_PATH = None

OUTPUT_DIR = None

# ---------------------------------------------
# REQUIRED prospective / operational parameters
# ---------------------------------------------
# Scientifically unresolved in the supplied knowledge base. Required before fit.
# Allowed: "float32" or "float64".
# Prospectively frozen before any preprocessing fit.
# float32 is used consistently for all transformed matrices and is suitable for
# downstream AE/DAE computation while reducing memory/storage pressure.
TRANSFORMED_DTYPE = "float32"

# Operational, not a scientific parameter, but it must be explicitly supplied
# rather than silently invented. Positive integer number of master rows per chunk.
# Explicit operational configuration for deterministic, memory-bounded I/O.
# This is NOT a learned preprocessing parameter and is intentionally excluded
# from the substantive preprocessing-policy hash.
MATERIALIZATION_CHUNK_ROWS = 50_000

# Parquet is the explicit storage format used by this implementation.
# REQUIRED operational configuration: set a PyArrow-supported codec string
# (for example one prospectively selected by the user), or the literal string
# "none" to explicitly request uncompressed Parquet. None means UNRESOLVED.
# Explicit operational storage choice. This does not alter the transformed
# numerical values, row membership, feature order, or corruption semantics.
PARQUET_COMPRESSION = "snappy"

# ------------------------------------------------------------------
# REQUIRED PROSPECTIVE UNSW CATEGORICAL POLICY — DO NOT LEAVE UNSET
# ------------------------------------------------------------------
# RARE_RULE_TYPE: "absolute_count" or "relative_frequency"
# Prospectively frozen minimal rare-category policy:
# group only benign-training singleton categories.
RARE_RULE_TYPE = "absolute_count"
# RARE_THRESHOLD: positive integer for absolute_count; (0,1) float for relative_frequency
RARE_THRESHOLD = 2
# Comparator: "lt" means support < threshold; "le" means support <= threshold
RARE_COMPARATOR = "lt"
# Required only for relative_frequency: "all_rows" or "nonmissing_rows"
RARE_RELATIVE_DENOMINATOR = None  # N/A for absolute_count

# Explicit reserved category representations. They must be distinct and must not
# collide with any raw category observed in any frozen experimental partition.
MISSING_CATEGORY_TOKEN = "__PRE02_MISSING__"
RARE_CATEGORY_TOKEN = "__PRE02_RARE__"
UNKNOWN_CATEGORY_TOKEN = "__PRE02_UNKNOWN__"

# Output location is resolved after the real Drive/project root is verified.
DEFAULT_OUTPUT_SUBDIR = "preprocessing_representation_s12_s14_v5_source_multiplicity_direct_pre00_20260920"

EXPERIMENTAL_PARTITIONS = [
    "benign_weight_train",
    "benign_early_stop",
    "benign_threshold_calibration",
    "development",
    "final_test",
]
OPTIONAL_GOVERNANCE_STATE = "attack_reserve_unused"

print("Dataset:", DATASET_NAME)
print("Stage:", NOTEBOOK_STAGE)
print("TRANSFORMED_DTYPE:", TRANSFORMED_DTYPE)
print("MATERIALIZATION_CHUNK_ROWS:", MATERIALIZATION_CHUNK_ROWS)


# Corrected branch binding.
CORRECTED_PRE01_SUBDIR = 'partitioning_s8_s11_v3_source_multiplicity_seed20260914'
PARENT_PRE01_SUBDIR = 'partitioning_s8_s11_v2_source_stratified_seed20260914'
EXPECTED_V3_MEMBERSHIP_SHA256 = '273093382be4d417a7b45c837008b994e57644902dffa1194002d2f2d0f8d6de'
EXPECTED_V3_POLICY_HASH = '17122c41630be246c40d9018a97d6f80b58d7422f438673594acb74fd4d2d3fb'
EXPECTED_V3_IMPLEMENTATION = 'v3.4_singleton_tolerance_fix'


Dataset: UNSW-NB15
Stage: PRE-02_SOURCE_MULTIPLICITY_V3
TRANSFORMED_DTYPE: float32
MATERIALIZATION_CHUNK_ROWS: 50000


In [2]:
# ============================================================
# 4. REQUIRED PROSPECTIVE CONFIGURATION CHECK — UNSW-NB15
# ============================================================
# Required prospective settings are frozen in the editable configuration cell before fitting.

required_values = {
    "TRANSFORMED_DTYPE": TRANSFORMED_DTYPE,
    "MATERIALIZATION_CHUNK_ROWS": MATERIALIZATION_CHUNK_ROWS,
    "PARQUET_COMPRESSION": PARQUET_COMPRESSION,
    "RARE_RULE_TYPE": RARE_RULE_TYPE,
    "RARE_THRESHOLD": RARE_THRESHOLD,
    "RARE_COMPARATOR": RARE_COMPARATOR,
    "MISSING_CATEGORY_TOKEN": MISSING_CATEGORY_TOKEN,
    "RARE_CATEGORY_TOKEN": RARE_CATEGORY_TOKEN,
    "UNKNOWN_CATEGORY_TOKEN": UNKNOWN_CATEGORY_TOKEN,
}
if RARE_RULE_TYPE == "relative_frequency":
    required_values["RARE_RELATIVE_DENOMINATOR"] = RARE_RELATIVE_DENOMINATOR

unresolved = [k for k, v in required_values.items() if v is None]

if unresolved:
    print(
        "PREPROCESSING / REPRESENTATION CONFIGURATION: FAIL"
    )
    print(
        "Unresolved prospective/operational configuration:",
        unresolved,
    )
    raise RuntimeError(
        "Required prospective/operational configuration remains unresolved "
        f"before PRE-02 materialization/fitting: {unresolved}"
    )
else:
    print(
        "PASS: all required prospective/operational PRE-02 configuration "
        "is explicitly frozen."
    )
    print("Unresolved prospective/operational configuration: None")


PASS: all required prospective/operational PRE-02 configuration is explicitly frozen.
Unresolved prospective/operational configuration: None


In [3]:
# ============================================================
# 5. OPTIONAL GOOGLE DRIVE MOUNT + VERIFIED PROJECT ROOT
# ============================================================
# A stale local /content/drive/MyDrive directory is not accepted as evidence that
# Google Drive is mounted. The real project must be visible before paths are bound.

import os
from pathlib import Path

PRIMARY_MOUNT = Path("/content/drive")
ALT_MOUNT_BASE = Path("/content/exp1v2_pre02_drive")


def _find_mydrive(mountpoint):
    mountpoint = Path(mountpoint)
    for name in ["MyDrive", "My Drive"]:
        p = mountpoint / name
        if p.exists() and p.is_dir():
            return p
    return None


def _find_project(mydrive, project_name):
    if mydrive is None:
        return None
    direct = Path(mydrive) / project_name
    if direct.exists() and direct.is_dir():
        return direct.resolve()

    matches = []
    for dirpath, dirnames, _ in os.walk(str(mydrive), topdown=True, followlinks=True):
        dirnames[:] = [d for d in dirnames if d not in {
            ".Trash", ".trash", "__pycache__", ".ipynb_checkpoints"
        }]
        if project_name in dirnames:
            matches.append((Path(dirpath) / project_name).resolve())

    unique = []
    seen = set()
    for p in matches:
        if str(p) not in seen:
            unique.append(p)
            seen.add(str(p))
    if len(unique) == 1:
        return unique[0]
    if len(unique) > 1:
        raise RuntimeError(
            f"Multiple {project_name!r} roots were found: {[str(p) for p in unique]}. "
            "Set PROJECT_ROOT explicitly."
        )
    return None


def _fresh_mountpoint(base):
    base = Path(base)
    for i in range(20):
        p = base if i == 0 else Path(f"{base}_{i}")
        if not p.exists() or (p.is_dir() and not any(p.iterdir())):
            return p
    raise RuntimeError("No fresh alternate Google Drive mountpoint is available.")


if PROJECT_ROOT is not None:
    PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
    if not PROJECT_ROOT.exists():
        raise FileNotFoundError(f"Configured PROJECT_ROOT does not exist: {PROJECT_ROOT}")
    DRIVE_SEARCH_ROOT = PROJECT_ROOT.parent
else:
    primary_mydrive = _find_mydrive(PRIMARY_MOUNT)
    project = _find_project(primary_mydrive, PROJECT_FOLDER_NAME)

    if project is None:
        try:
            from google.colab import drive
        except ImportError as exc:
            raise RuntimeError(
                "No verified project is visible and Google Colab Drive mounting is unavailable. "
                "Set PROJECT_ROOT explicitly."
            ) from exc

        alt = _fresh_mountpoint(ALT_MOUNT_BASE)
        print("Mounting Google Drive at verified alternate mountpoint:", alt)
        drive.mount(str(alt), force_remount=False)
        alt_mydrive = _find_mydrive(alt)
        project = _find_project(alt_mydrive, PROJECT_FOLDER_NAME)
        if project is None:
            raise FileNotFoundError(
                f"Google Drive mounted at {alt}, but {PROJECT_FOLDER_NAME!r} was not found. "
                "Set PROJECT_ROOT explicitly if the project directory has a different name."
            )
        DRIVE_SEARCH_ROOT = alt.resolve()
    else:
        DRIVE_SEARCH_ROOT = PRIMARY_MOUNT.resolve()

    PROJECT_ROOT = project.resolve()

DATASET_ROOT = PROJECT_ROOT / "DATASETS" / DATASET_FOLDER
if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Dataset root does not exist: {DATASET_ROOT}")

# Bind the corrected PRE-01 explicitly so the superseded PASS manifest cannot create discovery ambiguity.
if PRE01_INTEGRITY_REPORT_PATH is None:
    PRE01_INTEGRITY_REPORT_PATH = DATASET_ROOT / CORRECTED_PRE01_SUBDIR / "partition_integrity_report.json"

if OUTPUT_DIR is None:
    OUTPUT_DIR = DATASET_ROOT / DEFAULT_OUTPUT_SUBDIR
else:
    OUTPUT_DIR = Path(OUTPUT_DIR).expanduser()
    if not OUTPUT_DIR.is_absolute():
        OUTPUT_DIR = PROJECT_ROOT / OUTPUT_DIR
    OUTPUT_DIR = OUTPUT_DIR.resolve()

print("Verified PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_ROOT:", DATASET_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)


Mounting Google Drive at verified alternate mountpoint: /content/exp1v2_pre02_drive
Mounted at /content/exp1v2_pre02_drive
Verified PROJECT_ROOT: /content/exp1v2_pre02_drive/MyDrive/EXPERIMENT1_V2
DATASET_ROOT: /content/exp1v2_pre02_drive/MyDrive/EXPERIMENT1_V2/DATASETS/UNSW-NB15
OUTPUT_DIR: /content/exp1v2_pre02_drive/MyDrive/EXPERIMENT1_V2/DATASETS/UNSW-NB15/preprocessing_representation_s12_s14_v5_source_multiplicity_direct_pre00_20260920


In [4]:
# ============================================================
# 6. IMPORTS AND SOFTWARE ENVIRONMENT
# ============================================================
import sys
import gc
import json
import math
import hashlib
import platform
import tempfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import joblib
import sklearn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.preprocessing import OneHotEncoder


EXECUTION_UTC = datetime.now(timezone.utc).isoformat()
ENVIRONMENT_INFO = {
    "python": sys.version,
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "joblib": joblib.__version__,
    "pyarrow": pa.__version__,
    "platform": platform.platform(),
    "execution_utc": EXECUTION_UTC,
}
display(pd.DataFrame([{k: v for k, v in ENVIRONMENT_INFO.items() if k != "execution_utc"}]))


,python,pandas,numpy,scikit_learn,joblib,pyarrow,platform
0,"3.13.15 (main, Aug 6 2026, 11:06:22) [GCC 13....",2.2.3,2.1.3,1.6.1,1.6.0,23.0.1,Linux-6.6.122+-x86_64-with-glibc2.39


In [5]:
# ============================================================
# 7. UTILITY / HASH / SERIALIZATION FUNCTIONS
# ============================================================
BLOCKING_FAILURES = []
NONBLOCKING_WARNINGS = []
ARTIFACT_RECORDS = []


def canonical_json_bytes(obj):
    return json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def sha256_file(path, block_size=16 * 1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            block = f.read(block_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def canonical_hash_without_fields(obj, excluded_fields):
    body = dict(obj)
    for key in excluded_fields:
        body.pop(key, None)
    return sha256_bytes(canonical_json_bytes(body))


def read_json_required(path, description):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required {description}: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_json_new(path, obj):
    path = Path(path)
    if path.exists():
        raise RuntimeError(
            f"Refusing to overwrite a frozen PRE-02 artifact: {path}. "
            "Use a fresh/versioned OUTPUT_DIR for a new run."
        )
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = json.dumps(
        obj, indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False
    ) + "\n"
    path.write_text(payload, encoding="utf-8")
    return sha256_file(path)


def write_text_new(path, text):
    path = Path(path)
    if path.exists():
        raise RuntimeError(f"Refusing to overwrite frozen artifact: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(str(text), encoding="utf-8")
    return sha256_file(path)


def write_csv_new(path, df):
    path = Path(path)
    if path.exists():
        raise RuntimeError(f"Refusing to overwrite frozen artifact: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return sha256_file(path)


def write_parquet_new(path, df, compression=None):
    path = Path(path)
    if path.exists():
        raise RuntimeError(f"Refusing to overwrite frozen artifact: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False, engine="pyarrow", compression=compression)
    return sha256_file(path)


def register_artifact(path, purpose, section):
    path = Path(path)
    if not path.exists():
        return
    ARTIFACT_RECORDS.append({
        "artifact_name": path.name,
        "artifact_path": str(path),
        "artifact_type": path.suffix.lower().lstrip(".") or "file",
        "file_size": int(path.stat().st_size),
        "sha256": sha256_file(path),
        "purpose": purpose,
        "section": str(section),
    })


def record_warning(message):
    message = str(message)
    if message not in NONBLOCKING_WARNINGS:
        NONBLOCKING_WARNINGS.append(message)
    print("[NONBLOCKING WARNING]", message)


def hard_fail(message):
    message = str(message)
    BLOCKING_FAILURES.append(message)
    globals()["FINAL_STATUS"] = "FAIL"
    globals()["NEXT_AUTHORIZED_STAGE"] = "NONE"
    print("PREPROCESSING / REPRESENTATION INTEGRITY: FAIL")
    print("Next authorized stage: NONE")
    raise RuntimeError(message)


def require(condition, message):
    if not bool(condition):
        hard_fail(message)


def _dataset_matches(value):
    value = str(value).strip().casefold().replace("_", "-")
    expected = DATASET_NAME.casefold().replace("_", "-")
    aliases = {expected}
    if DATASET_NAME == "CICIDS2017":
        aliases |= {"cicids-2017", "cicids2017"}
    else:
        aliases |= {"unsw-nb15", "unswnb15"}
    return value in aliases


def _unique_paths(paths):
    out, seen = [], set()
    for p in paths:
        p = Path(p).resolve()
        if str(p) not in seen:
            out.append(p)
            seen.add(str(p))
    return out


def find_exact_files(root, filename):
    root = Path(root)
    return _unique_paths(p for p in root.rglob(filename) if p.is_file())


def resolve_one_exact(root, filename, description):
    matches = find_exact_files(root, filename)
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one {description} named {filename!r} under {root}; "
            f"found {[str(p) for p in matches]}"
        )
    return matches[0]


def canonical_membership_hash(df):
    required = ["row_id", "partition", "feature_fingerprint"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"Canonical membership hash missing columns: {missing}")
    ordered = df[required].sort_values("row_id", kind="mergesort")
    h = hashlib.sha256()
    for row in ordered.itertuples(index=False, name=None):
        for value in row:
            b = str(value).encode("utf-8", errors="surrogatepass")
            h.update(len(b).to_bytes(8, "big"))
            h.update(b)
        h.update(b"\n")
    return h.hexdigest()


def canonical_string_list_hash(values, sort_values=False):
    values = [str(v) for v in values]
    if sort_values:
        values = sorted(values)
    h = hashlib.sha256()
    for value in values:
        b = value.encode("utf-8", errors="surrogatepass")
        h.update(len(b).to_bytes(8, "big"))
        h.update(b)
    return h.hexdigest()


def canonical_alignment_hash(df, position_column):
    required = ["row_id", "partition", position_column]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"Alignment hash missing columns: {missing}")
    h = hashlib.sha256()
    for row in df[required].itertuples(index=False, name=None):
        for value in row:
            b = str(value).encode("utf-8", errors="surrogatepass")
            h.update(len(b).to_bytes(8, "big"))
            h.update(b)
        h.update(b"\n")
    return h.hexdigest()


def parquet_num_rows(path):
    return int(pq.ParquetFile(path).metadata.num_rows)


def parquet_columns(path):
    return list(pq.ParquetFile(path).schema_arrow.names)


def iter_parquet_frames(path, columns, batch_rows):
    pf = pq.ParquetFile(path)
    for batch in pf.iter_batches(batch_size=int(batch_rows), columns=list(columns)):
        yield batch.to_pandas()


In [6]:
# ============================================================
# 8. UPSTREAM ARTIFACT DISCOVERY — ACCEPTED V3 + DIRECT PRE-00 BINDING
# ============================================================
MASTER_MANIFEST_FILENAME = 'UNSW_NB15_master_manifest.json'

def _valid_pre01_candidate(integrity_path):
    try:
        report = read_json_required(integrity_path, "PRE-01 v3 integrity report")
    except Exception:
        return None
    if not _dataset_matches(report.get("dataset")) or report.get("status") != "PASS":
        return None
    if report.get("implementation_version") != EXPECTED_V3_IMPLEMENTATION:
        return None
    if report.get("formal_notebook01_authorized") is not False:
        return None

    root = Path(integrity_path).resolve().parent
    required = {
        "integrity": root / "partition_integrity_report.json",
        "policy": root / "source_multiplicity_amendment.json",
        "membership": root / "partition_membership.parquet",
        "handoff": root / "preprocessing_handoff_manifest.json",
    }
    if not all(p.exists() for p in required.values()):
        return None

    membership_sha = sha256_file(required["membership"]).lower()
    if membership_sha != str(report.get("partition_membership_sha256", "")).lower():
        return None
    if membership_sha != EXPECTED_V3_MEMBERSHIP_SHA256.lower():
        return None

    policy = read_json_required(required["policy"], "v3 source-multiplicity amendment")
    policy_hash = canonical_hash_without_fields(policy, ["policy_hash"])
    if policy_hash.lower() != str(policy.get("policy_hash", "")).lower():
        return None
    if policy_hash.lower() != str(report.get("scientific_policy_hash", "")).lower():
        return None
    if policy_hash.lower() != EXPECTED_V3_POLICY_HASH.lower():
        return None

    handoff = read_json_required(required["handoff"], "v3 preprocessing handoff")
    handoff_hash = canonical_hash_without_fields(handoff, ["handoff_hash"])
    if handoff_hash.lower() != str(handoff.get("handoff_hash", "")).lower():
        return None
    if str(handoff.get("partition_membership_sha256", "")).lower() != membership_sha:
        return None
    if str(handoff.get("scientific_policy_hash", "")).lower() != policy_hash.lower():
        return None

    report_hash = canonical_hash_without_fields(report, ["created_at_utc", "manifest_hash"])
    if report_hash.lower() != str(report.get("manifest_hash", "")).lower():
        return None

    return {
        "root": root,
        "report": report,
        "paths": required,
        "policy": policy,
        "handoff": handoff,
    }


def discover_pre01(dataset_root, explicit_integrity=None):
    if explicit_integrity is not None:
        item = _valid_pre01_candidate(Path(explicit_integrity))
        if item is None:
            raise RuntimeError(
                "Configured PRE01_INTEGRITY_REPORT_PATH is not the accepted v3.4 partition."
            )
        return item

    preferred = dataset_root / CORRECTED_PRE01_SUBDIR / "partition_integrity_report.json"
    item = _valid_pre01_candidate(preferred)
    if item is None:
        raise RuntimeError(
            f"Accepted PRE-01 v3.4 run is missing or invalid: {preferred}"
        )
    return item


def ordered_row_fingerprint_digest(path, batch_rows=200_000):
    """
    Deterministic ordered digest of exactly (row_id, feature_fingerprint).
    Used only to prove that PRE-01 v3.4 and PRE-00 refer to the identical
    row/model-view fingerprint identity universe.
    """
    path = Path(path)
    pf = pq.ParquetFile(path)
    needed = ["row_id", "feature_fingerprint"]
    require(
        set(needed).issubset(set(pf.schema_arrow.names)),
        f"Missing row/fingerprint identity columns in {path}"
    )

    h = hashlib.sha256()
    rows = 0
    for batch in pf.iter_batches(batch_size=int(batch_rows), columns=needed):
        tab = pa.Table.from_batches([batch])
        row_ids = tab["row_id"].to_pylist()
        fps = tab["feature_fingerprint"].to_pylist()
        require(len(row_ids) == len(fps), "Identity-batch length mismatch.")
        for rid, fp in zip(row_ids, fps):
            a = str(rid).encode("utf-8")
            b = str(fp).encode("utf-8")
            h.update(len(a).to_bytes(4, "little", signed=False))
            h.update(a)
            h.update(len(b).to_bytes(4, "little", signed=False))
            h.update(b)
        rows += len(row_ids)
    return h.hexdigest(), int(rows)


def _pre00_root_from_readiness(path):
    path = Path(path).resolve()
    return path.parent.parent if path.parent.name == "readiness" else path.parent


def _collect_pre00(root):
    names = [
        "prepartition_readiness_report.json",
        MASTER_MANIFEST_FILENAME,
        "taxonomy_manifest.json",
        "row_identity_manifest.json",
        "raw_feature_policy.json",
        "fingerprint_audit.json",
        "row_modelview_group_mapping.parquet",
    ]
    out = {}
    for name in names:
        matches = find_exact_files(root, name)
        if len(matches) != 1:
            return None
        out[name] = matches[0]
    return out


def _valid_pre00_candidate(readiness_path, expected_upstream_hashes):
    try:
        readiness = read_json_required(readiness_path, "PRE-00 readiness report")
    except Exception:
        return None
    if not _dataset_matches(readiness.get("dataset")):
        return None
    if readiness.get("final_status") not in {"PASS", "PASS_WITH_NONBLOCKING_WARNINGS"}:
        return None
    if any(readiness.get(f"section_{i}_status") != "PASS" for i in range(1, 8)):
        return None

    for key in [
        "master_dataset_sha256",
        "global_protocol_hash",
        "taxonomy_hash",
        "row_identity_specification_hash",
        "raw_feature_policy_hash",
        "fingerprint_manifest_hash",
    ]:
        expected = str(expected_upstream_hashes.get(key, "")).lower()
        observed = str(readiness.get(key, "")).lower()
        if expected and observed != expected:
            return None

    root = _pre00_root_from_readiness(readiness_path)
    paths = _collect_pre00(root)
    if paths is None:
        return None
    if sha256_file(paths["row_modelview_group_mapping.parquet"]).lower() != str(
        readiness.get("fingerprint_manifest_hash", "")
    ).lower():
        return None
    return {"root": root, "readiness": readiness, "paths": paths}


def discover_pre00(dataset_root, expected_upstream_hashes, explicit_readiness=None):
    if explicit_readiness is not None:
        item = _valid_pre00_candidate(Path(explicit_readiness), expected_upstream_hashes)
        if item is None:
            raise RuntimeError(
                "Configured PRE00_READINESS_REPORT_PATH does not identify a complete, "
                "hash-consistent PRE-00 run."
            )
        return item

    candidates = []
    for p in find_exact_files(dataset_root, "prepartition_readiness_report.json"):
        item = _valid_pre00_candidate(p, expected_upstream_hashes)
        if item is not None:
            candidates.append(item)
    by_root = {str(x["root"]): x for x in candidates}
    candidates = list(by_root.values())
    if len(candidates) != 1:
        raise RuntimeError(
            "Unable to identify exactly one PRE-00 run that matches the selected PRE-01 "
            f"upstream hashes. Candidates={[str(x['root']) for x in candidates]}. "
            "Set PRE00_READINESS_REPORT_PATH explicitly."
        )
    return candidates[0]


def resolve_master_by_manifest(project_root, dataset_root, manifest, expected_sha, explicit=None):
    expected_sha = str(expected_sha).lower()
    if explicit is not None:
        p = Path(explicit).expanduser().resolve()
        if not p.exists():
            raise FileNotFoundError(f"Configured MASTER_DATASET_PATH is missing: {p}")
        if sha256_file(p).lower() != expected_sha:
            raise RuntimeError("Configured master SHA-256 does not match frozen PRE-00 identity.")
        return p

    name = manifest.get("master_filename")
    if not name:
        raise RuntimeError("Master manifest does not contain master_filename.")
    matches = find_exact_files(dataset_root, str(name))
    if not matches:
        matches = find_exact_files(project_root, str(name))
    matching = [p for p in matches if sha256_file(p).lower() == expected_sha]
    matching = _unique_paths(matching)
    if len(matching) != 1:
        raise RuntimeError(
            f"Expected one frozen master named {name!r} with SHA {expected_sha}; "
            f"found {[str(p) for p in matching]}. Set MASTER_DATASET_PATH explicitly."
        )
    return matching[0]


def resolve_global_protocol(project_root, expected_hash, explicit=None):
    expected_hash = str(expected_hash).lower()
    candidates = []
    if explicit is not None:
        candidates = [Path(explicit).expanduser().resolve()]
    else:
        candidates = find_exact_files(project_root, "experiment1_v2_global_protocol.json")
    matching = []
    for p in candidates:
        if not p.exists():
            continue
        obj = read_json_required(p, "global protocol")
        computed = canonical_hash_without_fields(obj, ["canonical_protocol_hash"])
        stored = str(obj.get("canonical_protocol_hash", "")).lower()
        if computed.lower() == stored == expected_hash:
            matching.append(p)
    matching = _unique_paths(matching)
    if len(matching) != 1:
        raise RuntimeError(
            "Unable to resolve exactly one global protocol matching the frozen canonical hash. "
            f"Matches={[str(p) for p in matching]}"
        )
    return matching[0]


def discover_pre00_by_v3_membership(dataset_root, v3_membership_path, explicit_readiness=None):
    """
    Resolve PRE-00 without depending on the superseded v2 PRE-01 directory.
    Candidate PRE-00 runs must pass their own readiness/hash checks AND their
    full ordered (row_id, feature_fingerprint) mapping must match PRE-01 v3.4.
    """
    v3_digest, v3_rows = ordered_row_fingerprint_digest(v3_membership_path)

    readiness_candidates = (
        [Path(explicit_readiness).expanduser().resolve()]
        if explicit_readiness is not None
        else find_exact_files(dataset_root, "prepartition_readiness_report.json")
    )

    matched = []
    for readiness_path in readiness_candidates:
        item = _valid_pre00_candidate(readiness_path, expected_upstream_hashes={})
        if item is None:
            continue

        mapping = item["paths"]["row_modelview_group_mapping.parquet"]
        pre00_digest, pre00_rows = ordered_row_fingerprint_digest(mapping)
        if pre00_rows != v3_rows or pre00_digest != v3_digest:
            continue

        readiness = item["readiness"]
        identity = (
            str(readiness.get("master_dataset_sha256", "")).lower(),
            str(readiness.get("global_protocol_hash", "")).lower(),
            str(readiness.get("taxonomy_hash", "")).lower(),
            str(readiness.get("row_identity_specification_hash", "")).lower(),
            str(readiness.get("raw_feature_policy_hash", "")).lower(),
            str(readiness.get("fingerprint_manifest_hash", "")).lower(),
        )
        item = dict(item)
        item["scientific_identity"] = identity
        item["row_fingerprint_binding_digest"] = v3_digest
        item["row_fingerprint_binding_rows"] = v3_rows
        matched.append(item)

    require(
        len(matched) > 0,
        "No PRE-00 run is directly identity-compatible with the accepted PRE-01 v3.4 "
        "membership. The required criterion is an exact full-dataset ordered match of "
        "(row_id, feature_fingerprint), not the presence of an archived v2 PRE-01 folder."
    )

    identities = {}
    for item in matched:
        identities.setdefault(item["scientific_identity"], []).append(item)

    require(
        len(identities) == 1,
        "More than one distinct PRE-00 scientific identity matches the v3 membership. "
        "Set PRE00_READINESS_REPORT_PATH explicitly after investigating the duplicate identities."
    )

    copies = next(iter(identities.values()))
    copies = sorted(copies, key=lambda x: str(x["root"]))
    chosen = copies[0]
    if len(copies) > 1:
        record_warning(
            f"Found {len(copies)} path copies of the same PRE-00 scientific identity; "
            f"using deterministic path {chosen['root']}."
        )
    return chosen


In [7]:
# ============================================================
# 9. RESOLVE ACCEPTED PRE-01 V3.4 AND DIRECTLY BIND TO PRE-00
# ============================================================
PRE01_DISCOVERY = discover_pre01(
    DATASET_ROOT,
    explicit_integrity=PRE01_INTEGRITY_REPORT_PATH,
)
PRE01_INTEGRITY = PRE01_DISCOVERY["report"]
PRE01_ROOT = PRE01_DISCOVERY["root"]

PRE01_INTEGRITY_REPORT_PATH = Path(
    PRE01_INTEGRITY_REPORT_PATH or PRE01_DISCOVERY["paths"]["integrity"]
)
PARTITION_POLICY_MANIFEST_PATH = Path(
    PARTITION_POLICY_MANIFEST_PATH or PRE01_DISCOVERY["paths"]["policy"]
)
PARTITION_MEMBERSHIP_PATH = Path(
    PARTITION_MEMBERSHIP_PATH or PRE01_DISCOVERY["paths"]["membership"]
)
PREPROCESSING_HANDOFF_MANIFEST_PATH = Path(
    PREPROCESSING_HANDOFF_MANIFEST_PATH or PRE01_DISCOVERY["paths"]["handoff"]
)

# Scientific lineage is proven directly from v3 membership to PRE-00.
# The historical parent-v2 SHA remains recorded in the v3 integrity report,
# but the superseded parent-v2 directory is not an operational dependency.
PRE00_DISCOVERY = discover_pre00_by_v3_membership(
    DATASET_ROOT,
    v3_membership_path=PARTITION_MEMBERSHIP_PATH,
    explicit_readiness=PRE00_READINESS_REPORT_PATH,
)
PRE00_ROOT = PRE00_DISCOVERY["root"]
PRE00_READINESS = PRE00_DISCOVERY["readiness"]
V3_PRE00_ROW_FP_BINDING_DIGEST = PRE00_DISCOVERY["row_fingerprint_binding_digest"]
V3_PRE00_ROW_FP_BINDING_ROWS = PRE00_DISCOVERY["row_fingerprint_binding_rows"]

PRE00_READINESS_REPORT_PATH = Path(
    PRE00_READINESS_REPORT_PATH
    or PRE00_DISCOVERY["paths"]["prepartition_readiness_report.json"]
)
MASTER_MANIFEST_PATH = Path(
    MASTER_MANIFEST_PATH or PRE00_DISCOVERY["paths"][MASTER_MANIFEST_FILENAME]
)
TAXONOMY_MANIFEST_PATH = Path(
    TAXONOMY_MANIFEST_PATH or PRE00_DISCOVERY["paths"]["taxonomy_manifest.json"]
)
ROW_IDENTITY_MANIFEST_PATH = Path(
    ROW_IDENTITY_MANIFEST_PATH or PRE00_DISCOVERY["paths"]["row_identity_manifest.json"]
)
RAW_FEATURE_POLICY_PATH = Path(
    RAW_FEATURE_POLICY_PATH or PRE00_DISCOVERY["paths"]["raw_feature_policy.json"]
)
FINGERPRINT_AUDIT_PATH = Path(
    FINGERPRINT_AUDIT_PATH or PRE00_DISCOVERY["paths"]["fingerprint_audit.json"]
)
FINGERPRINT_ROW_MAPPING_PATH = Path(
    FINGERPRINT_ROW_MAPPING_PATH
    or PRE00_DISCOVERY["paths"]["row_modelview_group_mapping.parquet"]
)

MASTER_MANIFEST = read_json_required(MASTER_MANIFEST_PATH, "master manifest")
MASTER_DATASET_PATH = resolve_master_by_manifest(
    PROJECT_ROOT,
    DATASET_ROOT,
    MASTER_MANIFEST,
    PRE00_READINESS["master_dataset_sha256"],
    explicit=MASTER_DATASET_PATH,
)
GLOBAL_PROTOCOL_PATH = resolve_global_protocol(
    PROJECT_ROOT,
    PRE00_READINESS["global_protocol_hash"],
    explicit=GLOBAL_PROTOCOL_PATH,
)

print("Accepted PRE-01 v3.4 root:", PRE01_ROOT)
print("PRE-00 root:", PRE00_ROOT)
print("Direct v3↔PRE-00 row/fingerprint binding rows:", V3_PRE00_ROW_FP_BINDING_ROWS)
print("Direct v3↔PRE-00 row/fingerprint binding digest:", V3_PRE00_ROW_FP_BINDING_DIGEST)
print("Historical parent-v2 membership SHA (record only):",
      PRE01_INTEGRITY.get("parent_membership_sha256"))
print("Master dataset:", MASTER_DATASET_PATH)


Accepted PRE-01 v3.4 root: /content/exp1v2_pre02_drive/MyDrive/EXPERIMENT1_V2/DATASETS/UNSW-NB15/partitioning_s8_s11_v3_source_multiplicity_seed20260914
PRE-00 root: /content/exp1v2_pre02_drive/MyDrive/EXPERIMENT1_V2/DATASETS/UNSW-NB15/UNSW_Audit_Governance
Direct v3↔PRE-00 row/fingerprint binding rows: 2540047
Direct v3↔PRE-00 row/fingerprint binding digest: e84ab027e5d78f23f1e2ee8b56d5e0c7f178952d3553c74de14df667189eca0d
Historical parent-v2 membership SHA (record only): 3114e887aeff514be147d68a9f938d9c889d2c89a21a1ccffd6ff65fcb4a4fac
Master dataset: /content/exp1v2_pre02_drive/MyDrive/EXPERIMENT1_V2/DATASETS/UNSW-NB15/MERGED/UNSW-NB15_merged_canonical.csv


In [8]:
# ============================================================
# 10–11. HARD V3.4 ↔ PRE-00 READINESS / IDENTITY GATE
# ============================================================
GLOBAL_PROTOCOL = read_json_required(GLOBAL_PROTOCOL_PATH, "global protocol")
TAXONOMY_MANIFEST = read_json_required(TAXONOMY_MANIFEST_PATH, "taxonomy manifest")
ROW_IDENTITY_MANIFEST = read_json_required(ROW_IDENTITY_MANIFEST_PATH, "row identity manifest")
RAW_FEATURE_POLICY = read_json_required(RAW_FEATURE_POLICY_PATH, "raw feature policy")
FINGERPRINT_AUDIT = read_json_required(FINGERPRINT_AUDIT_PATH, "fingerprint audit")
PARTITION_POLICY = read_json_required(
    PARTITION_POLICY_MANIFEST_PATH, "v3 source-multiplicity amendment"
)
PREPROCESSING_HANDOFF = read_json_required(
    PREPROCESSING_HANDOFF_MANIFEST_PATH, "v3 preprocessing handoff"
)

ALLOWED_STATUS = {"PASS", "PASS_WITH_NONBLOCKING_WARNINGS"}

# PRE-00 and PRE-01 readiness.
require(
    PRE00_READINESS.get("final_status") in ALLOWED_STATUS,
    "PRE-00 readiness is not PASS/WARNING."
)
require(
    PRE01_INTEGRITY.get("status") == "PASS",
    "Accepted PRE-01 v3.4 integrity is not PASS."
)
require(
    PRE01_INTEGRITY.get("implementation_version") == EXPECTED_V3_IMPLEMENTATION,
    "Unexpected PRE-01 implementation version."
)
require(
    PRE01_INTEGRITY.get("formal_notebook01_authorized") is False,
    "PRE-01 unexpectedly authorizes formal modeling."
)
require(
    PRE01_INTEGRITY.get("final_test_status") == "FROZEN_AND_EVALUATION_SEALED",
    "Final test is not sealed upstream."
)

for gate_name, gate_value in PRE01_INTEGRITY.get("gates", {}).items():
    if gate_name == "fingerprint_crossings":
        require(int(gate_value) == 0, "PRE-01 v3 fingerprint crossing gate failed.")
    else:
        require(gate_value is True, f"PRE-01 v3 gate is not PASS: {gate_name}")

for s in range(1, 8):
    require(
        PRE00_READINESS.get(f"section_{s}_status") == "PASS",
        f"PRE-00 Section {s} is not PASS."
    )

if PRE00_READINESS.get("final_status") == "PASS_WITH_NONBLOCKING_WARNINGS":
    record_warning(
        "PRE-00 passed with nonblocking warnings; warnings are preserved upstream."
    )

# Recompute upstream identities from the chosen PRE-00.
MASTER_SHA256 = sha256_file(MASTER_DATASET_PATH)
GLOBAL_PROTOCOL_HASH = canonical_hash_without_fields(
    GLOBAL_PROTOCOL, ["canonical_protocol_hash"]
)
RAW_FEATURE_POLICY_HASH = canonical_hash_without_fields(
    RAW_FEATURE_POLICY, ["feature_policy_hash"]
)
PARTITION_POLICY_HASH = canonical_hash_without_fields(
    PARTITION_POLICY, ["policy_hash"]
)
PARTITION_MANIFEST_FILE_SHA256 = sha256_file(PARTITION_MEMBERSHIP_PATH)
FINGERPRINT_MANIFEST_HASH = sha256_file(FINGERPRINT_ROW_MAPPING_PATH)

_membership_for_hash = pd.read_parquet(
    PARTITION_MEMBERSHIP_PATH,
    columns=["row_id", "partition", "feature_fingerprint"],
)
PARTITION_MANIFEST_CANONICAL_HASH = canonical_membership_hash(
    _membership_for_hash
)
del _membership_for_hash
gc.collect()

TAXONOMY_HASH = str(TAXONOMY_MANIFEST.get("taxonomy_hash", "")).lower()
ROW_IDENTITY_SPEC_HASH = str(
    ROW_IDENTITY_MANIFEST.get("row_identity_specification_hash", "")
).lower()

checks = []
def add_check(name, observed, expected):
    ok = (
        str(observed).lower() == str(expected).lower()
        and str(expected) != ""
    )
    checks.append({
        "identity": name,
        "observed": str(observed),
        "expected": str(expected),
        "status": "PASS" if ok else "FAIL",
    })
    require(
        ok,
        f"Upstream identity mismatch for {name}: "
        f"observed={observed}, expected={expected}"
    )

# PRE-00 self-consistency.
add_check(
    "master_dataset_sha256",
    MASTER_SHA256,
    PRE00_READINESS.get("master_dataset_sha256"),
)
add_check(
    "master_manifest.master_sha256",
    MASTER_SHA256,
    MASTER_MANIFEST.get("master_sha256"),
)
add_check(
    "global_protocol_hash",
    GLOBAL_PROTOCOL_HASH,
    PRE00_READINESS.get("global_protocol_hash"),
)
add_check(
    "stored_global_protocol_hash",
    GLOBAL_PROTOCOL_HASH,
    GLOBAL_PROTOCOL.get("canonical_protocol_hash"),
)
add_check(
    "taxonomy_hash",
    TAXONOMY_HASH,
    PRE00_READINESS.get("taxonomy_hash"),
)
add_check(
    "row_identity_specification_hash",
    ROW_IDENTITY_SPEC_HASH,
    PRE00_READINESS.get("row_identity_specification_hash"),
)
add_check(
    "raw_feature_policy_hash",
    RAW_FEATURE_POLICY_HASH,
    PRE00_READINESS.get("raw_feature_policy_hash"),
)
add_check(
    "stored_raw_feature_policy_hash",
    RAW_FEATURE_POLICY_HASH,
    RAW_FEATURE_POLICY.get("feature_policy_hash"),
)
add_check(
    "fingerprint_manifest_hash",
    FINGERPRINT_MANIFEST_HASH,
    PRE00_READINESS.get("fingerprint_manifest_hash"),
)

# Accepted v3 partition identity.
add_check(
    "v3_scientific_policy_hash",
    PARTITION_POLICY_HASH,
    PRE01_INTEGRITY.get("scientific_policy_hash"),
)
add_check(
    "stored_v3_policy_hash",
    PARTITION_POLICY_HASH,
    PARTITION_POLICY.get("policy_hash"),
)
add_check(
    "expected_v3_policy_hash",
    PARTITION_POLICY_HASH,
    EXPECTED_V3_POLICY_HASH,
)
add_check(
    "v3_partition_file_sha256",
    PARTITION_MANIFEST_FILE_SHA256,
    PRE01_INTEGRITY.get("partition_membership_sha256"),
)
add_check(
    "expected_v3_partition_file_sha256",
    PARTITION_MANIFEST_FILE_SHA256,
    EXPECTED_V3_MEMBERSHIP_SHA256,
)
add_check(
    "handoff_partition_file_sha256",
    PARTITION_MANIFEST_FILE_SHA256,
    PREPROCESSING_HANDOFF.get("partition_membership_sha256"),
)
add_check(
    "handoff_scientific_policy_hash",
    PARTITION_POLICY_HASH,
    PREPROCESSING_HANDOFF.get("scientific_policy_hash"),
)

# Direct PRE-01 v3.4 ↔ PRE-00 scientific lineage proof.
_v3_digest_check, _v3_digest_rows = ordered_row_fingerprint_digest(
    PARTITION_MEMBERSHIP_PATH
)
_pre00_digest_check, _pre00_digest_rows = ordered_row_fingerprint_digest(
    FINGERPRINT_ROW_MAPPING_PATH
)
add_check(
    "v3_pre00_ordered_row_fingerprint_digest",
    _v3_digest_check,
    _pre00_digest_check,
)
require(
    _v3_digest_rows == _pre00_digest_rows == V3_PRE00_ROW_FP_BINDING_ROWS,
    "v3/PRE-00 row-fingerprint binding row counts disagree."
)
require(
    _v3_digest_check == V3_PRE00_ROW_FP_BINDING_DIGEST,
    "v3/PRE-00 direct-binding digest changed after discovery."
)

# Internal governance bindings.
require(
    RAW_FEATURE_POLICY.get("feature_policy_status") == "PASS",
    "Raw feature policy is not PASS."
)
require(
    ROW_IDENTITY_MANIFEST.get("validation_status") == "PASS",
    "Row identity manifest is not PASS."
)
require(
    str(FINGERPRINT_AUDIT.get("master_dataset_sha256", "")).lower()
    == MASTER_SHA256.lower(),
    "Fingerprint audit is bound to another master."
)
require(
    str(RAW_FEATURE_POLICY.get("master_dataset_sha256", "")).lower()
    == MASTER_SHA256.lower(),
    "Raw feature policy is bound to another master."
)
require(
    str(TAXONOMY_MANIFEST.get("master_dataset_sha256", "")).lower()
    == MASTER_SHA256.lower(),
    "Taxonomy is bound to another master."
)

# Handoff restrictions.
require(
    PREPROCESSING_HANDOFF.get("fit_preprocessing_on")
    == "benign_weight_train only",
    "v3 handoff does not restrict fit to benign_weight_train."
)
require(
    PREPROCESSING_HANDOFF.get("transform_roles") == EXPERIMENTAL_PARTITIONS,
    "v3 handoff transform-role list differs from the frozen five roles."
)
require(
    PREPROCESSING_HANDOFF.get("formal_notebook01_authorized") is False,
    "v3 handoff prematurely authorizes Notebook 01."
)
require(
    "no final-test outcome access"
    in str(PREPROCESSING_HANDOFF.get("final_test_rule", "")).lower(),
    "v3 handoff does not preserve final-test sealing."
)

UPSTREAM_HASHES = {
    "master_dataset_sha256": MASTER_SHA256,
    "global_protocol_hash": GLOBAL_PROTOCOL_HASH,
    "taxonomy_hash": TAXONOMY_HASH,
    "row_identity_specification_hash": ROW_IDENTITY_SPEC_HASH,
    "raw_feature_policy_hash": RAW_FEATURE_POLICY_HASH,
    "fingerprint_manifest_hash": FINGERPRINT_MANIFEST_HASH,
    "partition_policy_hash": PARTITION_POLICY_HASH,
    "partition_manifest_canonical_hash": PARTITION_MANIFEST_CANONICAL_HASH,
    "partition_manifest_file_sha256": PARTITION_MANIFEST_FILE_SHA256,
    "v3_pre00_ordered_row_fingerprint_digest": V3_PRE00_ROW_FP_BINDING_DIGEST,
    "v3_pre00_ordered_row_fingerprint_rows": V3_PRE00_ROW_FP_BINDING_ROWS,
    "historical_parent_v2_membership_sha256_declared":
        PRE01_INTEGRITY.get("parent_membership_sha256"),
}

UPSTREAM_CHECK_DF = pd.DataFrame(checks)
UPSTREAM_READINESS_DF = pd.DataFrame([
    {"stage": "PRE-00", "status": PRE00_READINESS.get("final_status")},
    {"stage": "PRE-01-v3.4", "status": PRE01_INTEGRITY.get("status")},
])

display(UPSTREAM_READINESS_DF)
display(UPSTREAM_CHECK_DF[["identity", "status"]])
print(
    "PASS: accepted PRE-01 v3.4 is directly bound to PRE-00 by the "
    "full ordered row/fingerprint identity universe; archived v2 PRE-01 "
    "is not required operationally."
)


[NONBLOCKING WARNING] PRE-00 passed with nonblocking warnings; warnings are preserved upstream.


,stage,status
0,PRE-00,PASS_WITH_NONBLOCKING_WARNINGS
1,PRE-01-v3.4,PASS


,identity,status
0,master_dataset_sha256,PASS
1,master_manifest.master_sha256,PASS
2,global_protocol_hash,PASS
3,stored_global_protocol_hash,PASS
4,taxonomy_hash,PASS
5,row_identity_specification_hash,PASS
6,raw_feature_policy_hash,PASS
7,stored_raw_feature_policy_hash,PASS
8,fingerprint_manifest_hash,PASS
9,v3_scientific_policy_hash,PASS


PASS: accepted PRE-01 v3.4 is directly bound to PRE-00 by the full ordered row/fingerprint identity universe; archived v2 PRE-01 is not required operationally.


In [9]:
# ============================================================
# 12. LOAD IMMUTABLE MEMBERSHIP + FREEZE THE RAW MODEL-VIEW CONTRACT
# ============================================================
MODEL_INPUT_COLUMNS = list(RAW_FEATURE_POLICY.get("column_order", []))
INCLUDED_MODEL_COLUMNS = list(RAW_FEATURE_POLICY.get("included_model_columns", []))
NUMERICAL_MODEL_COLUMNS = list(RAW_FEATURE_POLICY.get("numerical_model_columns", []))
CATEGORICAL_MODEL_COLUMNS = list(RAW_FEATURE_POLICY.get("categorical_model_columns", []))
METADATA_COLUMNS_DECLARED = list(RAW_FEATURE_POLICY.get("metadata_columns", []))

require(MODEL_INPUT_COLUMNS, "Frozen raw feature policy has an empty column_order.")
require(MODEL_INPUT_COLUMNS == INCLUDED_MODEL_COLUMNS, "Frozen included_model_columns and column_order disagree.")
require(len(MODEL_INPUT_COLUMNS) == len(set(MODEL_INPUT_COLUMNS)), "Frozen model-input columns contain duplicates.")

# UNSW-specific frozen feature-policy assertions.
def _norm_name(x):
    return "".join(ch.lower() for ch in str(x) if ch.isalnum())
_cat_norms = [_norm_name(c) for c in CATEGORICAL_MODEL_COLUMNS]
require(_cat_norms == ["proto", "state", "service"], f"UNSW frozen categorical columns must be proto/state/service in that order; got {CATEGORICAL_MODEL_COLUMNS}")
_prohibited = {"srcip", "dstip", "sport", "dsport", "stime", "ltime", "label", "attackcatoriginal", "attackcatcanonical"}
_accidental = [c for c in MODEL_INPUT_COLUMNS if _norm_name(c) in _prohibited]
require(not _accidental, f"Prohibited UNSW fields remain in frozen model inputs: {_accidental}")
_ct_ftp_matches = [c for c in MODEL_INPUT_COLUMNS if _norm_name(c) == "ctftpcmd"]
if _ct_ftp_matches:
    require(all(c in NUMERICAL_MODEL_COLUMNS for c in _ct_ftp_matches),
            f"Frozen ct_ftp_cmd field must remain numerical; got {_ct_ftp_matches}")
    require(not any(c in CATEGORICAL_MODEL_COLUMNS for c in _ct_ftp_matches),
            "ct_ftp_cmd must not be silently reclassified as categorical.")
if len(NUMERICAL_MODEL_COLUMNS) != 38:
    record_warning(
        f"Verified frozen UNSW numerical feature count is {len(NUMERICAL_MODEL_COLUMNS)}, not the prior expectation of 38; the verified frozen count is retained."
    )

_expected_corruption = RAW_FEATURE_POLICY.get("expected_future_corruption_structure", {})
require(str(_expected_corruption.get("numerical_masking", "")).lower() == "searched later",
        "Frozen UNSW feature policy does not authorize later numerical-masking search as expected.")
_grouped = _expected_corruption.get("grouped_categorical_masking", {})
require(isinstance(_grouped, dict) and {_norm_name(k) for k in _grouped.keys()} == {"proto", "state", "service"},
        "Frozen UNSW feature policy grouped categorical corruption structure does not contain exactly proto/state/service.")


MEMBERSHIP = pd.read_parquet(PARTITION_MEMBERSHIP_PATH)
required_membership_columns = {
    "row_id", "feature_fingerprint", "fingerprint_group_class", "partition",
    "binary_label", "attack_family", "source_file", "source_row_number",
    "reserve_reason_if_applicable",
}
missing_membership = sorted(required_membership_columns - set(MEMBERSHIP.columns))
require(not missing_membership, f"Partition membership is missing columns: {missing_membership}")
require(not MEMBERSHIP["row_id"].isna().any(), "Membership contains null row_id values.")
require(not MEMBERSHIP["row_id"].duplicated().any(), "Membership contains duplicate row_id values.")
require(set(EXPERIMENTAL_PARTITIONS).issubset(set(MEMBERSHIP["partition"].astype(str).unique())), "One or more required experimental partitions are absent.")
require(set(MEMBERSHIP["partition"].astype(str).unique()).issubset(set(EXPERIMENTAL_PARTITIONS + [OPTIONAL_GOVERNANCE_STATE])), "Unexpected PRE-01 membership state exists.")

MASTER_ROW_COUNT = int(MASTER_MANIFEST.get("row_count", -1))
require(MASTER_ROW_COUNT > 0, "Master manifest row_count is invalid.")
require(len(MEMBERSHIP) == MASTER_ROW_COUNT, "Membership row count does not equal frozen master row count.")
require(parquet_num_rows(FINGERPRINT_ROW_MAPPING_PATH) == MASTER_ROW_COUNT, "Fingerprint row mapping row count does not equal master row count.")
require(int(ROW_IDENTITY_MANIFEST.get("row_count", -1)) == MASTER_ROW_COUNT, "Row identity row count does not equal master row count.")

train_attack_count = int(
    MEMBERSHIP.loc[
        MEMBERSHIP["partition"].astype(str).eq("benign_weight_train"),
        "binary_label",
    ].astype(int).sum()
)
require(train_attack_count == 0, f"benign_weight_train contains {train_attack_count} attack rows.")

PARTITION_COUNTS = {
    p: int((MEMBERSHIP["partition"].astype(str) == p).sum())
    for p in EXPERIMENTAL_PARTITIONS
}

feature_table = pd.DataFrame([
    {"feature_class": "raw model inputs", "count": len(MODEL_INPUT_COLUMNS)},
    {"feature_class": "numerical", "count": len(NUMERICAL_MODEL_COLUMNS)},
    {"feature_class": "categorical", "count": len(CATEGORICAL_MODEL_COLUMNS)},
])
display(feature_table)
display(pd.DataFrame([{"partition": p, "rows": PARTITION_COUNTS[p]} for p in EXPERIMENTAL_PARTITIONS]))


,feature_class,count
0,raw model inputs,41
1,numerical,38
2,categorical,3


,partition,rows
0,benign_weight_train,1109381
1,benign_early_stop,221878
2,benign_threshold_calibration,332815
3,development,350389
4,final_test,525584


In [10]:
# ============================================================
# 13. RAW-MATERIALIZATION UTILITIES
# ============================================================
def validate_materialization_operational_config():
    if MATERIALIZATION_CHUNK_ROWS is None:
        hard_fail("MATERIALIZATION_CHUNK_ROWS is unresolved. Set a positive integer prospectively before materialization.")
    try:
        n = int(MATERIALIZATION_CHUNK_ROWS)
    except Exception as exc:
        raise RuntimeError("MATERIALIZATION_CHUNK_ROWS must be an integer.") from exc
    require(n > 0, "MATERIALIZATION_CHUNK_ROWS must be positive.")

    require(PARQUET_COMPRESSION is not None,
            "PARQUET_COMPRESSION is unresolved. Explicitly configure a PyArrow codec or the literal string 'none'.")
    codec = str(PARQUET_COMPRESSION).strip().lower()
    require(bool(codec), "PARQUET_COMPRESSION must not be empty.")
    if codec in {"none", "uncompressed"}:
        effective_codec = None
    else:
        try:
            available = bool(pa.Codec.is_available(codec))
        except Exception:
            available = False
        require(available, f"Configured Parquet compression codec is unavailable in this runtime: {codec!r}")
        effective_codec = codec

    require(TRANSFORMED_DTYPE in {"float32", "float64"},
            "TRANSFORMED_DTYPE is unresolved/invalid. Prospectively set float32 or float64 before any PRE-02 output is materialized.")
    _early_required = {
        "RARE_RULE_TYPE": RARE_RULE_TYPE,
        "RARE_THRESHOLD": RARE_THRESHOLD,
        "RARE_COMPARATOR": RARE_COMPARATOR,
        "MISSING_CATEGORY_TOKEN": MISSING_CATEGORY_TOKEN,
        "RARE_CATEGORY_TOKEN": RARE_CATEGORY_TOKEN,
        "UNKNOWN_CATEGORY_TOKEN": UNKNOWN_CATEGORY_TOKEN,
    }
    if RARE_RULE_TYPE == "relative_frequency":
        _early_required["RARE_RELATIVE_DENOMINATOR"] = RARE_RELATIVE_DENOMINATOR
    _early_missing = [k for k, v in _early_required.items() if v is None]
    require(not _early_missing,
            f"Required prospective UNSW categorical configuration is unresolved before materialization: {_early_missing}")

    return n, effective_codec, codec


def build_master_ordinal_routing(membership, row_identity_manifest):
    ranges = row_identity_manifest.get("source_ranges")
    require(isinstance(ranges, list) and ranges, "Row identity manifest lacks source_ranges.")

    start_map = {str(r["source_file"]): int(r["merged_start_0based"]) for r in ranges}
    count_map = {str(r["source_file"]): int(r["row_count"]) for r in ranges}
    require(len(start_map) == len(ranges), "Duplicate source_file entries exist in row identity source ranges.")

    source = membership["source_file"].astype("string")
    source_row = pd.to_numeric(membership["source_row_number"], errors="coerce")
    require(not source_row.isna().any(), "Membership source_row_number contains nonnumeric values.")
    source_row = source_row.astype("int64")

    unknown_sources = sorted(set(source.astype(str).unique()) - set(start_map))
    require(not unknown_sources, f"Membership refers to source files absent from row identity ranges: {unknown_sources}")

    starts = source.astype(str).map(start_map).astype("int64")
    source_counts = source.astype(str).map(count_map).astype("int64")
    require(bool(((source_row >= 1) & (source_row <= source_counts)).all()), "Membership source-row number falls outside frozen source ranges.")

    ordinals = starts.to_numpy(dtype=np.int64) + source_row.to_numpy(dtype=np.int64) - 1
    require(int(ordinals.min()) == 0, "Minimum reconstructed master ordinal is not zero.")
    require(int(ordinals.max()) == MASTER_ROW_COUNT - 1, "Maximum reconstructed master ordinal does not reach the final master row.")
    require(len(np.unique(ordinals)) == MASTER_ROW_COUNT, "Reconstructed master ordinals are not a complete one-to-one mapping.")

    # Validate the frozen row_id construction rule in bounded slices.
    step = int(MATERIALIZATION_CHUNK_ROWS)
    for start in range(0, len(membership), step):
        part = membership.iloc[start:start + step]
        expected = (
            DATASET_PREFIX + "::" + part["source_file"].astype(str)
            + "::" + part["source_row_number"].astype(int).map(lambda x: f"{x:010d}")
        )
        bad = part["row_id"].astype(str).to_numpy() != expected.to_numpy()
        require(not bool(np.any(bad)), "Membership row_id disagrees with the frozen PRE-00 source-provenance identity rule.")

    partition_codes = {p: i for i, p in enumerate(EXPERIMENTAL_PARTITIONS)}
    partition_codes[OPTIONAL_GOVERNANCE_STATE] = len(EXPERIMENTAL_PARTITIONS)
    codes = membership["partition"].astype(str).map(partition_codes)
    require(not codes.isna().any(), "Unable to encode one or more membership states.")

    route = np.full(MASTER_ROW_COUNT, -1, dtype=np.int8)
    route[ordinals] = codes.to_numpy(dtype=np.int8)
    require(not bool(np.any(route < 0)), "At least one master ordinal has no PRE-01 membership state.")

    out = membership.copy()
    out["row_ordinal_0based"] = ordinals
    if not out["row_ordinal_0based"].is_monotonic_increasing:
        out = out.sort_values("row_ordinal_0based", kind="mergesort").reset_index(drop=True)
    else:
        out = out.reset_index(drop=True)
    out["raw_row_position"] = out.groupby("partition", sort=False).cumcount().astype("int64")
    return route, out, partition_codes


def iter_master_model_chunks(path, columns, chunksize):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        dtype_map = {c: "string" for c in columns}
        for chunk in pd.read_csv(
            path,
            usecols=list(columns),
            chunksize=int(chunksize),
            low_memory=False,
            dtype=dtype_map,
            keep_default_na=False,
            na_filter=False,
        ):
            yield chunk[list(columns)]
    elif suffix in {".parquet", ".pq"}:
        pf = pq.ParquetFile(path)
        for batch in pf.iter_batches(batch_size=int(chunksize), columns=list(columns)):
            yield batch.to_pandas()[list(columns)]
    else:
        raise RuntimeError(f"Unsupported frozen master format: {path}")


def prepare_master_chunk_for_raw(chunk):
    # UNSW raw materialization preserves lexical raw values. Numerical parsing and
    # whitespace-only structural-missing normalization occur only in Section 12 preprocessing.
    out = pd.DataFrame(index=chunk.index)
    for col in MODEL_INPUT_COLUMNS:
        out[col] = chunk[col].astype("string")
    return out[MODEL_INPUT_COLUMNS]


class _PartitionParquetWriters:
    def __init__(self, output_paths, compression=None):
        self.paths = {k: Path(v) for k, v in output_paths.items()}
        self.compression = compression
        self.writers = {k: None for k in self.paths}
        self.schemas = {k: None for k in self.paths}
        self.rows = {k: 0 for k in self.paths}
        for p in self.paths.values():
            if p.exists():
                raise RuntimeError(f"Refusing to overwrite existing raw partition: {p}")
            p.parent.mkdir(parents=True, exist_ok=True)

    def write(self, key, frame):
        if len(frame) == 0:
            return
        table = pa.Table.from_pandas(frame, preserve_index=False)
        if self.writers[key] is None:
            self.schemas[key] = table.schema
            self.writers[key] = pq.ParquetWriter(
                self.paths[key], table.schema, compression=self.compression
            )
        elif table.schema != self.schemas[key]:
            table = table.cast(self.schemas[key], safe=False)
        self.writers[key].write_table(table)
        self.rows[key] += len(frame)

    def close(self):
        for writer in self.writers.values():
            if writer is not None:
                writer.close()


In [11]:
# ============================================================
# 14. MATERIALIZE THE FIVE RAW EXPERIMENTAL PARTITIONS
# ============================================================
CHUNK_ROWS, PARQUET_COMPRESSION_EFFECTIVE, PARQUET_COMPRESSION_CONFIGURED = validate_materialization_operational_config()

# No upstream identity or partition mismatch has been tolerated to this point.
# Create the PRE-02 output tree only now.
OUTPUT_DIR = Path(OUTPUT_DIR)
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    hard_fail(
        f"OUTPUT_DIR already contains files: {OUTPUT_DIR}. Use a fresh/versioned output directory; PRE-02 will not overwrite frozen artifacts."
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR = OUTPUT_DIR / "raw_partitions"
TRANSFORMED_DIR = OUTPUT_DIR / "transformed_partitions"
MANIFEST_DIR = OUTPUT_DIR / "manifests"
PARAMETER_DIR = OUTPUT_DIR / "parameters"
REPORT_DIR = OUTPUT_DIR / "reports"
for d in [RAW_DIR, TRANSFORMED_DIR, MANIFEST_DIR, PARAMETER_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

ENVIRONMENT_PATH = MANIFEST_DIR / "environment.json"
write_json_new(ENVIRONMENT_PATH, ENVIRONMENT_INFO)
register_artifact(ENVIRONMENT_PATH, "PRE-02 software environment", "12")

ROUTE_BY_ORDINAL, MEMBERSHIP_ORDERED, PARTITION_CODE_MAP = build_master_ordinal_routing(
    MEMBERSHIP,
    ROW_IDENTITY_MANIFEST,
)

RAW_MODEL_PATHS = {
    p: RAW_DIR / f"{p}_raw.parquet"
    for p in EXPERIMENTAL_PARTITIONS
}
RAW_METADATA_PATHS = {
    p: RAW_DIR / f"{p}_raw_metadata.parquet"
    for p in EXPERIMENTAL_PARTITIONS
}

metadata_preferred = [
    "row_id", "partition", "binary_label", "attack_family",
    "feature_fingerprint", "fingerprint_group_class",
    "source_file", "source_row_number", "reserve_reason_if_applicable",
    "attack_cat_original", "attack_cat_canonical",
]
metadata_columns = [c for c in metadata_preferred if c in MEMBERSHIP_ORDERED.columns]

RAW_ALIGNMENT_HASHES = {}
for p in EXPERIMENTAL_PARTITIONS:
    side = MEMBERSHIP_ORDERED.loc[
        MEMBERSHIP_ORDERED["partition"].astype(str).eq(p),
        metadata_columns + ["raw_row_position"],
    ].copy()
    require(len(side) == PARTITION_COUNTS[p], f"Metadata sidecar row count mismatch for {p}.")
    require(np.array_equal(side["raw_row_position"].to_numpy(dtype=np.int64), np.arange(len(side), dtype=np.int64)), f"Non-contiguous raw row positions for {p}.")
    write_parquet_new(RAW_METADATA_PATHS[p], side, compression=PARQUET_COMPRESSION_EFFECTIVE)
    RAW_ALIGNMENT_HASHES[p] = canonical_alignment_hash(side, "raw_row_position")
    register_artifact(RAW_METADATA_PATHS[p], f"Raw metadata/provenance sidecar for {p}", "12")
    del side

# Optional attack reserve is materialized as governance metadata only.
ATTACK_RESERVE_METADATA_PATH = None
if OPTIONAL_GOVERNANCE_STATE in set(MEMBERSHIP_ORDERED["partition"].astype(str).unique()):
    reserve = MEMBERSHIP_ORDERED.loc[
        MEMBERSHIP_ORDERED["partition"].astype(str).eq(OPTIONAL_GOVERNANCE_STATE),
        metadata_columns + ["raw_row_position"],
    ].copy()
    if len(reserve):
        ATTACK_RESERVE_METADATA_PATH = RAW_DIR / "attack_reserve_unused_metadata.parquet"
        write_parquet_new(ATTACK_RESERVE_METADATA_PATH, reserve, compression=PARQUET_COMPRESSION_EFFECTIVE)
        register_artifact(ATTACK_RESERVE_METADATA_PATH, "Governance-only unused attack reserve metadata", "12")
    del reserve

writers = _PartitionParquetWriters(RAW_MODEL_PATHS, compression=PARQUET_COMPRESSION_EFFECTIVE)
code_to_partition = {PARTITION_CODE_MAP[p]: p for p in EXPERIMENTAL_PARTITIONS}
offset = 0
try:
    for chunk in iter_master_model_chunks(
        MASTER_DATASET_PATH,
        MODEL_INPUT_COLUMNS,
        CHUNK_ROWS,
    ):
        n = len(chunk)
        require(offset + n <= MASTER_ROW_COUNT, "Master reader exceeded frozen master row count.")
        prepared = prepare_master_chunk_for_raw(chunk)
        require(list(prepared.columns) == MODEL_INPUT_COLUMNS, "Prepared raw model-view column order changed.")
        codes = ROUTE_BY_ORDINAL[offset:offset + n]
        for code_value, p in code_to_partition.items():
            mask = codes == code_value
            if bool(np.any(mask)):
                writers.write(p, prepared.loc[mask, MODEL_INPUT_COLUMNS].reset_index(drop=True))
        offset += n
        del chunk, prepared, codes
        gc.collect()
finally:
    writers.close()

require(offset == MASTER_ROW_COUNT, f"Master materialization read {offset} rows; expected {MASTER_ROW_COUNT}.")
for p in EXPERIMENTAL_PARTITIONS:
    require(RAW_MODEL_PATHS[p].exists(), f"Raw model partition was not written: {p}")
    require(writers.rows[p] == PARTITION_COUNTS[p], f"Raw model row count mismatch for {p}: {writers.rows[p]} vs {PARTITION_COUNTS[p]}")
    require(parquet_num_rows(RAW_MODEL_PATHS[p]) == PARTITION_COUNTS[p], f"Written Parquet row count mismatch for {p}.")
    require(parquet_columns(RAW_MODEL_PATHS[p]) == MODEL_INPUT_COLUMNS, f"Raw model column order mismatch for {p}.")
    register_artifact(RAW_MODEL_PATHS[p], f"Materialized raw model-input partition {p}", "12")

RAW_MODEL_HASHES = {p: sha256_file(RAW_MODEL_PATHS[p]) for p in EXPERIMENTAL_PARTITIONS}
RAW_METADATA_HASHES = {p: sha256_file(RAW_METADATA_PATHS[p]) for p in EXPERIMENTAL_PARTITIONS}

print("PASS: five raw experimental partitions materialized strictly from frozen master + PRE-01 membership.")
display(pd.DataFrame([
    {"partition": p, "expected_rows": PARTITION_COUNTS[p], "materialized_rows": writers.rows[p]}
    for p in EXPERIMENTAL_PARTITIONS
]))


PASS: five raw experimental partitions materialized strictly from frozen master + PRE-01 membership.


,partition,expected_rows,materialized_rows
0,benign_weight_train,1109381,1109381
1,benign_early_stop,221878,221878
2,benign_threshold_calibration,332815,332815
3,development,350389,350389
4,final_test,525584,525584


In [12]:
# ============================================================
# 15. RAW MATERIALIZATION INTEGRITY + MODEL/METADATA SEPARATION GATE
# ============================================================
protected_metadata_names = {
    "row_id", "partition", "binary_label", "attack_family",
    "feature_fingerprint", "fingerprint_group_class",
    "source_file", "source_row_number", "reserve_reason_if_applicable",
    "attack_cat_original", "attack_cat_canonical", "Label",
}
protected_metadata_names |= set(str(x) for x in RAW_FEATURE_POLICY.get("excluded_columns", []))
for _m in RAW_FEATURE_POLICY.get("metadata_columns", []):
    _base = str(_m).split("(", 1)[0].strip()
    if _base:
        protected_metadata_names.add(_base)

raw_integrity_rows = []
for p in EXPERIMENTAL_PARTITIONS:
    model_cols = parquet_columns(RAW_MODEL_PATHS[p])
    meta_cols = parquet_columns(RAW_METADATA_PATHS[p])
    overlap = sorted(set(model_cols) & protected_metadata_names)
    require(not overlap, f"Protected metadata entered raw model matrix for {p}: {overlap}")
    require(model_cols == MODEL_INPUT_COLUMNS, f"Raw model-input schema/order mismatch for {p}.")
    require("row_id" in meta_cols and "partition" in meta_cols, f"Metadata sidecar is missing identity fields for {p}.")
    require(parquet_num_rows(RAW_MODEL_PATHS[p]) == parquet_num_rows(RAW_METADATA_PATHS[p]), f"Raw model/metadata row mismatch for {p}.")
    raw_integrity_rows.append({
        "partition": p,
        "rows": parquet_num_rows(RAW_MODEL_PATHS[p]),
        "model_columns": len(model_cols),
        "metadata_columns": len(meta_cols),
        "protected_metadata_in_X": len(overlap),
        "status": "PASS",
    })

RAW_INTEGRITY_DF = pd.DataFrame(raw_integrity_rows)
display(RAW_INTEGRITY_DF)
print("PASS: raw model matrices are separated from metadata/provenance sidecars.")


,partition,rows,model_columns,metadata_columns,protected_metadata_in_X,status
0,benign_weight_train,1109381,41,12,0,PASS
1,benign_early_stop,221878,41,12,0,PASS
2,benign_threshold_calibration,332815,41,12,0,PASS
3,development,350389,41,12,0,PASS
4,final_test,525584,41,12,0,PASS


PASS: raw model matrices are separated from metadata/provenance sidecars.


In [13]:
# ============================================================
# 16. DATASET-SPECIFIC DETERMINISTIC NUMERICAL HANDLING + RAW AUDIT
# ============================================================

def normalize_numeric_frame(frame):
    out = pd.DataFrame(index=frame.index)
    audit = []
    for col in NUMERICAL_MODEL_COLUMNS:
        s = frame[col]
        text = s.astype("string")
        pandas_missing = s.isna()
        whitespace = text.notna() & text.str.strip().eq("")
        missing_like = pandas_missing | whitespace
        prepared = text.mask(missing_like, pd.NA)
        parsed = pd.to_numeric(prepared, errors="coerce")
        invalid = (~missing_like) & parsed.isna()
        if bool(invalid.any()):
            examples = s.loc[invalid].astype(str).head(5).tolist()
            hard_fail(f"UNSW frozen numerical column {col!r} contains genuine nonmissing nonnumeric text. Examples={examples}")
        arr = parsed.to_numpy(dtype=np.float64, na_value=np.nan)
        posinf = np.isposinf(arr)
        neginf = np.isneginf(arr)
        if bool(posinf.any() or neginf.any()):
            hard_fail(
                f"UNSW frozen numerical column {col!r} contains +/-Inf, but no upstream Inf-to-missing policy is frozen."
            )
        out[col] = parsed.astype("float64")
        audit.append({
            "feature": col,
            "raw_missing": int(pandas_missing.sum()),
            "whitespace_missing": int(whitespace.sum()),
            "positive_inf": int(posinf.sum()),
            "negative_inf": int(neginf.sum()),
            "invalid_text": int(invalid.sum()),
            "post_deterministic_missing": int(parsed.isna().sum()),
        })
    return out[NUMERICAL_MODEL_COLUMNS], audit


NUMERICAL_AUDIT_ROWS = []
for p in EXPERIMENTAL_PARTITIONS:
    totals = {c: {
        "raw_missing": 0, "whitespace_missing": 0,
        "positive_inf": 0, "negative_inf": 0,
        "invalid_text": 0, "post_deterministic_missing": 0,
    } for c in NUMERICAL_MODEL_COLUMNS}
    for frame in iter_parquet_frames(RAW_MODEL_PATHS[p], NUMERICAL_MODEL_COLUMNS, CHUNK_ROWS):
        normalized, audit = normalize_numeric_frame(frame)
        for row in audit:
            col = row.pop("feature")
            for k, v in row.items():
                totals[col][k] += int(v)
        del frame, normalized
        gc.collect()
    for col in NUMERICAL_MODEL_COLUMNS:
        NUMERICAL_AUDIT_ROWS.append({"partition": p, "feature": col, **totals[col]})

NUMERICAL_AUDIT_DF = pd.DataFrame(NUMERICAL_AUDIT_ROWS)
NUMERICAL_AUDIT_PATH = REPORT_DIR / "numerical_raw_audit.csv"
write_csv_new(NUMERICAL_AUDIT_PATH, NUMERICAL_AUDIT_DF)
register_artifact(NUMERICAL_AUDIT_PATH, "Complete raw numerical missing/Inf/validity audit", "12")

aggregate = NUMERICAL_AUDIT_DF.groupby("partition", as_index=False)[[
    "raw_missing", "whitespace_missing", "positive_inf", "negative_inf",
    "invalid_text", "post_deterministic_missing",
]].sum()
# Preserve final-test diagnostic blindness in the visible notebook. Complete mechanical
# counts remain frozen in numerical_raw_audit.csv and are not used for decisions.
aggregate_display = aggregate.astype("object").copy()
final_mask = aggregate_display["partition"].astype(str).eq("final_test")
for _c in ["raw_missing", "whitespace_missing", "positive_inf", "negative_inf",
           "invalid_text", "post_deterministic_missing"]:
    aggregate_display.loc[final_mask, _c] = "SEALED_IN_ARTIFACT"
display(aggregate_display)


,partition,raw_missing,whitespace_missing,positive_inf,negative_inf,invalid_text,post_deterministic_missing
0,benign_early_stop,0,332761,0,0,0,332761
1,benign_threshold_calibration,0,497666,0,0,0,497666
2,benign_weight_train,0,1663856,0,0,0,1663856
3,development,0,686314,0,0,0,686314
4,final_test,SEALED_IN_ARTIFACT,SEALED_IN_ARTIFACT,SEALED_IN_ARTIFACT,SEALED_IN_ARTIFACT,SEALED_IN_ARTIFACT,SEALED_IN_ARTIFACT


In [14]:
# ============================================================
# 17. UNSW PROSPECTIVE CATEGORICAL CONFIGURATION GATE + TOKEN COLLISION AUDIT
# ============================================================
def validate_unsw_categorical_configuration():
    missing = []
    required = {
        "RARE_RULE_TYPE": RARE_RULE_TYPE,
        "RARE_THRESHOLD": RARE_THRESHOLD,
        "RARE_COMPARATOR": RARE_COMPARATOR,
        "MISSING_CATEGORY_TOKEN": MISSING_CATEGORY_TOKEN,
        "RARE_CATEGORY_TOKEN": RARE_CATEGORY_TOKEN,
        "UNKNOWN_CATEGORY_TOKEN": UNKNOWN_CATEGORY_TOKEN,
    }
    if RARE_RULE_TYPE == "relative_frequency":
        required["RARE_RELATIVE_DENOMINATOR"] = RARE_RELATIVE_DENOMINATOR
    for k, v in required.items():
        if v is None:
            missing.append(k)
    require(not missing, f"Required UNSW prospective categorical configuration is unresolved: {missing}")

    require(RARE_RULE_TYPE in {"absolute_count", "relative_frequency"}, "RARE_RULE_TYPE must be absolute_count or relative_frequency.")
    require(RARE_COMPARATOR in {"lt", "le"}, "RARE_COMPARATOR must be lt or le.")
    if RARE_RULE_TYPE == "absolute_count":
        require(isinstance(RARE_THRESHOLD, (int, np.integer)) and not isinstance(RARE_THRESHOLD, bool) and int(RARE_THRESHOLD) > 0, "Absolute rare threshold must be a positive integer.")
    else:
        require(isinstance(RARE_THRESHOLD, (int, float, np.integer, np.floating)) and 0.0 < float(RARE_THRESHOLD) < 1.0, "Relative rare threshold must be in (0,1).")
        require(RARE_RELATIVE_DENOMINATOR in {"all_rows", "nonmissing_rows"}, "Relative rare denominator must be all_rows or nonmissing_rows.")

    raw_tokens = [MISSING_CATEGORY_TOKEN, RARE_CATEGORY_TOKEN, UNKNOWN_CATEGORY_TOKEN]
    require(all(isinstance(x, str) and x != "" for x in raw_tokens),
            "Missing/rare/unknown category tokens must be explicit non-empty strings.")
    tokens = [str(x) for x in raw_tokens]
    require(len(set(tokens)) == 3, "Missing/rare/unknown category tokens must be distinct.")
    return tokens


RESERVED_CATEGORY_TOKENS = validate_unsw_categorical_configuration()

# Reserved-token compatibility is checked ONLY on benign_weight_train.
# Non-fit partitions, especially final_test, are not allowed to trigger redesign of
# the prospectively configured categorical representation. Unseen downstream raw
# strings are mapped through the already-frozen UNKNOWN policy.
collision_rows = []
for frame in iter_parquet_frames(RAW_MODEL_PATHS["benign_weight_train"], CATEGORICAL_MODEL_COLUMNS, CHUNK_ROWS):
    for col in CATEGORICAL_MODEL_COLUMNS:
        s = frame[col].astype("string")
        for token in RESERVED_CATEGORY_TOKENS:
            n = int(s.eq(token).fillna(False).sum())
            if n:
                collision_rows.append({"partition": "benign_weight_train", "feature": col, "token": token, "count": n})
    del frame
    gc.collect()
require(not collision_rows, f"Configured reserved categorical token collides with benign_weight_train raw category values: {collision_rows[:20]}")
UNSW_CATEGORICAL_POLICY_CONFIG_DF = pd.DataFrame([
    {"item": "rare_rule_type", "configured_value": RARE_RULE_TYPE},
    {"item": "rare_threshold", "configured_value": RARE_THRESHOLD},
    {"item": "rare_comparator", "configured_value": RARE_COMPARATOR},
    {"item": "rare_relative_denominator", "configured_value": RARE_RELATIVE_DENOMINATOR if RARE_RULE_TYPE == "relative_frequency" else "N/A"},
    {"item": "missing_token", "configured_value": str(MISSING_CATEGORY_TOKEN)},
    {"item": "rare_token", "configured_value": str(RARE_CATEGORY_TOKEN)},
    {"item": "unknown_token", "configured_value": str(UNKNOWN_CATEGORY_TOKEN)},
])
display(UNSW_CATEGORICAL_POLICY_CONFIG_DF)
print("PASS: configured missing/rare/unknown tokens are distinct and collision-free in benign_weight_train; non-fit partitions cannot redesign this policy.")


,item,configured_value
0,rare_rule_type,absolute_count
1,rare_threshold,2
2,rare_comparator,lt
3,rare_relative_denominator,N/A
4,missing_token,__PRE02_MISSING__
5,rare_token,__PRE02_RARE__
6,unknown_token,__PRE02_UNKNOWN__


PASS: configured missing/rare/unknown tokens are distinct and collision-free in benign_weight_train; non-fit partitions cannot redesign this policy.


In [15]:
# ============================================================
# 18. FREEZE PREPROCESSING POLICY MANIFEST + PRE-FIT LEAKAGE GATE
# ============================================================
def validate_prefit_configuration():
    require(TRANSFORMED_DTYPE in {"float32", "float64"}, "TRANSFORMED_DTYPE must be prospectively set to float32 or float64 before fitting.")
    require(isinstance(CHUNK_ROWS, int) and CHUNK_ROWS > 0, "Materialization chunk size is not valid.")
    require(PARQUET_COMPRESSION_CONFIGURED is not None, "Parquet compression configuration is unresolved.")
    validate_unsw_categorical_configuration()


validate_prefit_configuration()

# Hard prerequisites immediately before any learned .fit() call.
require(PRE00_READINESS.get("final_status") in ALLOWED_STATUS, "PRE-00 no longer passes readiness gate.")
require(PRE01_INTEGRITY.get("status") == "PASS", "Accepted PRE-01 v3.4 no longer passes integrity gate.")
require(int((MEMBERSHIP["partition"].astype(str) == "benign_weight_train").sum()) == PARTITION_COUNTS["benign_weight_train"], "benign_weight_train membership count changed.")
require(int(MEMBERSHIP.loc[MEMBERSHIP["partition"].astype(str).eq("benign_weight_train"), "binary_label"].astype(int).sum()) == 0, "benign_weight_train is not purely benign.")
for p in EXPERIMENTAL_PARTITIONS:
    require(RAW_INTEGRITY_DF.loc[RAW_INTEGRITY_DF["partition"].eq(p), "status"].iloc[0] == "PASS", f"Raw integrity gate failed for {p}.")

PREPROCESSING_POLICY_SUBSTANTIVE = {
    "dataset": DATASET_NAME,
    "protocol_version": PROTOCOL_VERSION,
    "notebook_stage": NOTEBOOK_STAGE,
    "upstream_hashes": UPSTREAM_HASHES,
    "raw_model_columns": MODEL_INPUT_COLUMNS,
    "numerical_columns": NUMERICAL_MODEL_COLUMNS,
    "categorical_columns": CATEGORICAL_MODEL_COLUMNS,
    "numerical_preprocessing": {

        "numerical_missing_policy": "pandas NA/NaN or whitespace-only raw token -> missing; genuine other nonnumeric text is blocking; +/-Inf is blocking because no Inf normalization policy is frozen",
        "imputation_method": "median",
        "scaling_method": "StandardScaler",
        "fit_partition": "benign_weight_train",
    },

        "categorical_preprocessing": {
            "columns": CATEGORICAL_MODEL_COLUMNS,
            "missing_recognition": "pandas NA/NaN only; nonmissing raw strings are preserved",
            "missing_token": str(MISSING_CATEGORY_TOKEN),
            "rare_rule_type": RARE_RULE_TYPE,
            "rare_threshold": float(RARE_THRESHOLD) if RARE_RULE_TYPE == "relative_frequency" else int(RARE_THRESHOLD),
            "rare_comparator": RARE_COMPARATOR,
            "rare_relative_denominator": RARE_RELATIVE_DENOMINATOR if RARE_RULE_TYPE == "relative_frequency" else "N/A",
            "rare_token": str(RARE_CATEGORY_TOKEN),
            "unknown_token": str(UNKNOWN_CATEGORY_TOKEN),
            "category_order_rule": "retained benign-training categories sorted lexicographically, then configured missing/rare/unknown tokens",
        },
        "encoder_type": "sklearn.preprocessing.OneHotEncoder with explicit benign-training-derived categories",
    "fit_partition": "benign_weight_train",
    "transform_partitions": EXPERIMENTAL_PARTITIONS,
    "final_test_rule": "transformation permitted; influence forbidden",
    "transformed_dtype": TRANSFORMED_DTYPE,
    "algorithm_version": "Experiment1_v2_PRE02_representation_v2_source_multiplicity",
}
# Operational serialization choices are deliberately excluded from the substantive
# preprocessing-policy hash: they may alter file bytes/row groups but not the
# learned representation or scientific preprocessing semantics.
OPERATIONAL_STORAGE_CONFIG = {
    "format": "parquet",
    "compression_configured": PARQUET_COMPRESSION_CONFIGURED,
    "compression_effective": (PARQUET_COMPRESSION_EFFECTIVE if PARQUET_COMPRESSION_EFFECTIVE is not None else "none"),
    "materialization_chunk_rows": int(CHUNK_ROWS),
}
PREPROCESSING_POLICY_HASH = sha256_bytes(canonical_json_bytes(PREPROCESSING_POLICY_SUBSTANTIVE))
PREPROCESSING_POLICY = {
    **PREPROCESSING_POLICY_SUBSTANTIVE,
    "operational_storage": OPERATIONAL_STORAGE_CONFIG,
    "created_at_utc": EXECUTION_UTC,
    "preprocessing_policy_hash": PREPROCESSING_POLICY_HASH,
}
PREPROCESSING_POLICY_PATH = MANIFEST_DIR / "preprocessing_policy_manifest.json"
write_json_new(PREPROCESSING_POLICY_PATH, PREPROCESSING_POLICY)
register_artifact(PREPROCESSING_POLICY_PATH, "Frozen PRE-02 preprocessing policy", "12")

print("PRE-FIT LEAKAGE GATE: PASS")
print("Preprocessing policy hash:", PREPROCESSING_POLICY_HASH)


PRE-FIT LEAKAGE GATE: PASS
Preprocessing policy hash: 610c536ba782613483afbf8650e94fe57913519d9ee168d1a152c8828a612c64


In [16]:
# ============================================================
# 19. FIT NUMERICAL MEDIAN IMPUTER + STANDARDSCALER ON benign_weight_train ONLY
# ============================================================
FIT_SOURCE_PARTITIONS_USED = {"benign_weight_train"}
TRAIN_RAW_PATH = RAW_MODEL_PATHS["benign_weight_train"]

numeric_chunks = []
for frame in iter_parquet_frames(TRAIN_RAW_PATH, NUMERICAL_MODEL_COLUMNS, CHUNK_ROWS):
    normalized, _ = normalize_numeric_frame(frame)
    numeric_chunks.append(normalized.to_numpy(dtype=np.float64, copy=True))
    del frame, normalized
    gc.collect()

require(numeric_chunks, "No benign_weight_train numerical rows were loaded for fitting.")
X_TRAIN_NUMERIC = np.vstack(numeric_chunks)
del numeric_chunks
gc.collect()
require(X_TRAIN_NUMERIC.shape == (PARTITION_COUNTS["benign_weight_train"], len(NUMERICAL_MODEL_COLUMNS)), "Training numerical matrix shape mismatch.")

all_missing = np.isnan(X_TRAIN_NUMERIC).all(axis=0)
require(not bool(all_missing.any()), f"One or more frozen numerical features are entirely missing in benign_weight_train: {[NUMERICAL_MODEL_COLUMNS[i] for i in np.flatnonzero(all_missing)]}")

NUMERICAL_IMPUTER = SimpleImputer(strategy="median", copy=True, add_indicator=False)
NUMERICAL_IMPUTER.fit(X_TRAIN_NUMERIC)
X_TRAIN_IMPUTED = NUMERICAL_IMPUTER.transform(X_TRAIN_NUMERIC)
require(np.isfinite(X_TRAIN_IMPUTED).all(), "Training numerical matrix remains non-finite after median imputation.")

NUMERICAL_SCALER = StandardScaler(copy=True, with_mean=True, with_std=True)
NUMERICAL_SCALER.fit(X_TRAIN_IMPUTED)

NUMERICAL_PARAMETER_DF = pd.DataFrame({
    "feature": NUMERICAL_MODEL_COLUMNS,
    "imputer_median": NUMERICAL_IMPUTER.statistics_,
    "scaler_mean": NUMERICAL_SCALER.mean_,
    "scaler_scale": NUMERICAL_SCALER.scale_,
    "scaler_var": NUMERICAL_SCALER.var_,
})
NUMERICAL_PARAMETER_PATH = PARAMETER_DIR / "numerical_fitted_parameters.csv"
write_csv_new(NUMERICAL_PARAMETER_PATH, NUMERICAL_PARAMETER_DF)
register_artifact(NUMERICAL_PARAMETER_PATH, "Training-fitted numerical imputation/scaling parameters", "12")

display(NUMERICAL_PARAMETER_DF.head(20))
print("Numerical fit source: benign_weight_train only")

# Release the large training arrays before categorical fitting / transformation.
del X_TRAIN_NUMERIC, X_TRAIN_IMPUTED
gc.collect()


,feature,imputer_median,scaler_mean,scaler_scale,scaler_var
0,dur,2.001400e-02,6.441169e-01,1.223744e+01,1.497550e+02
1,sbytes,1.684000e+03,4.107771e+03,1.055958e+04,1.115047e+08
2,dbytes,2.464000e+03,4.132148e+04,1.674008e+05,2.802302e+10
3,sttl,3.100000e+01,3.708727e+01,2.816224e+01,7.931117e+02
4,dttl,2.900000e+01,2.860429e+01,2.661043e+01,7.081151e+02
5,sloss,4.000000e+00,5.522054e+00,8.845679e+00,7.824603e+01
6,dloss,5.000000e+00,1.846910e+01,5.842600e+01,3.413598e+03
7,Sload,5.593869e+05,2.825270e+07,1.086525e+08,1.180536e+16
8,Dload,6.557031e+05,2.794507e+06,4.402325e+06,1.938046e+13
9,Spkts,1.400000e+01,3.680519e+01,6.752153e+01,4.559157e+03


Numerical fit source: benign_weight_train only


0

In [17]:
# ============================================================
# 20. UNSW TRAINING-ONLY RARE/VOCABULARY POLICY + ONE-HOT ENCODER FIT
# ============================================================
def _is_rare(count, denominator):
    if RARE_RULE_TYPE == "absolute_count":
        value = int(count)
        threshold = int(RARE_THRESHOLD)
    else:
        value = float(count) / float(denominator)
        threshold = float(RARE_THRESHOLD)
    return value < threshold if RARE_COMPARATOR == "lt" else value <= threshold

# Training-only support counts.
support = {c: Counter() for c in CATEGORICAL_MODEL_COLUMNS}
missing_counts = {c: 0 for c in CATEGORICAL_MODEL_COLUMNS}
train_rows_seen = 0
for frame in iter_parquet_frames(TRAIN_RAW_PATH, CATEGORICAL_MODEL_COLUMNS, CHUNK_ROWS):
    train_rows_seen += len(frame)
    for col in CATEGORICAL_MODEL_COLUMNS:
        s = frame[col].astype("string")
        missing_counts[col] += int(s.isna().sum())
        support[col].update(s.dropna().astype(str).tolist())
    del frame
gc.collect()
require(train_rows_seen == PARTITION_COUNTS["benign_weight_train"], "Categorical support scan row count mismatch.")

CATEGORICAL_POLICY = {}
support_rows = []
for col in CATEGORICAL_MODEL_COLUMNS:
    if RARE_RULE_TYPE == "relative_frequency":
        denominator = (
            train_rows_seen
            if RARE_RELATIVE_DENOMINATOR == "all_rows"
            else train_rows_seen - missing_counts[col]
        )
        require(denominator > 0, f"Relative rare-rule denominator is zero for {col}.")
    else:
        denominator = train_rows_seen

    rare = set()
    retained = set()
    for category, count in support[col].items():
        if _is_rare(count, denominator):
            rare.add(str(category))
            status = "RARE"
        else:
            retained.add(str(category))
            status = "RETAINED"
        support_rows.append({
            "feature": col,
            "category": str(category),
            "support": int(count),
            "relative_frequency_all_training_rows": float(count) / float(train_rows_seen),
            "training_policy_status": status,
        })

    categories = sorted(retained)
    categories.extend([
        str(MISSING_CATEGORY_TOKEN),
        str(RARE_CATEGORY_TOKEN),
        str(UNKNOWN_CATEGORY_TOKEN),
    ])
    require(len(categories) == len(set(categories)), f"Encoded category list contains duplicates for {col}.")

    CATEGORICAL_POLICY[col] = {
        "retained_categories": sorted(retained),
        "rare_categories": sorted(rare),
        "encoder_categories": categories,
        "training_missing_count": int(missing_counts[col]),
    }

CATEGORICAL_SUPPORT_DF = pd.DataFrame(support_rows)
CATEGORICAL_SUPPORT_PATH = PARAMETER_DIR / "categorical_training_support.csv"
write_csv_new(CATEGORICAL_SUPPORT_PATH, CATEGORICAL_SUPPORT_DF)
register_artifact(CATEGORICAL_SUPPORT_PATH, "Benign-training categorical support and rare-policy assignments", "12")

CATEGORICAL_POLICY_PATH = PARAMETER_DIR / "categorical_fitted_policy.json"
write_json_new(CATEGORICAL_POLICY_PATH, {
    "dataset": DATASET_NAME,
    "fit_partition": "benign_weight_train",
    "rare_rule_type": RARE_RULE_TYPE,
    "rare_threshold": RARE_THRESHOLD,
    "rare_comparator": RARE_COMPARATOR,
    "rare_relative_denominator": RARE_RELATIVE_DENOMINATOR if RARE_RULE_TYPE == "relative_frequency" else "N/A",
    "missing_token": str(MISSING_CATEGORY_TOKEN),
    "rare_token": str(RARE_CATEGORY_TOKEN),
    "unknown_token": str(UNKNOWN_CATEGORY_TOKEN),
    "per_feature": CATEGORICAL_POLICY,
})
register_artifact(CATEGORICAL_POLICY_PATH, "Frozen benign-training-derived categorical vocabulary policy", "12")


def map_categorical_frame(frame, collect_unknown=False):
    mapped = pd.DataFrame(index=frame.index)
    unknown_counts = {}
    for col in CATEGORICAL_MODEL_COLUMNS:
        s = frame[col].astype("string")
        is_missing = s.isna()
        rare_set = set(CATEGORICAL_POLICY[col]["rare_categories"])
        retained_set = set(CATEGORICAL_POLICY[col]["retained_categories"])

        out = s.astype("object")
        out.loc[is_missing] = str(MISSING_CATEGORY_TOKEN)
        nonmissing = ~is_missing
        rare_mask = nonmissing & s.isin(rare_set)
        retained_mask = nonmissing & s.isin(retained_set)
        unknown_mask = nonmissing & ~(rare_mask | retained_mask)
        out.loc[rare_mask] = str(RARE_CATEGORY_TOKEN)
        out.loc[unknown_mask] = str(UNKNOWN_CATEGORY_TOKEN)
        mapped[col] = out.astype(str)
        unknown_counts[col] = int(unknown_mask.sum()) if collect_unknown else 0
    return mapped[CATEGORICAL_MODEL_COLUMNS], unknown_counts

# Fit OneHotEncoder ONLY on benign_weight_train values after the benign-training-derived mapping.
cat_chunks = []
for frame in iter_parquet_frames(TRAIN_RAW_PATH, CATEGORICAL_MODEL_COLUMNS, CHUNK_ROWS):
    mapped, unknown_counts = map_categorical_frame(frame, collect_unknown=True)
    require(sum(unknown_counts.values()) == 0, "Unknown categories appeared while mapping benign_weight_train; categorical policy construction is inconsistent.")
    cat_chunks.append(mapped)
    del frame, mapped
    gc.collect()
TRAIN_CAT_MAPPED = pd.concat(cat_chunks, axis=0, ignore_index=True)
del cat_chunks
gc.collect()
require(len(TRAIN_CAT_MAPPED) == PARTITION_COUNTS["benign_weight_train"], "Training categorical matrix row count mismatch.")

explicit_categories = [CATEGORICAL_POLICY[c]["encoder_categories"] for c in CATEGORICAL_MODEL_COLUMNS]
try:
    CATEGORICAL_ENCODER = OneHotEncoder(
        categories=explicit_categories,
        handle_unknown="error",
        sparse_output=False,
        dtype=np.dtype(TRANSFORMED_DTYPE),
    )
except TypeError:
    # Compatibility with older scikit-learn while preserving the same dense semantics.
    CATEGORICAL_ENCODER = OneHotEncoder(
        categories=explicit_categories,
        handle_unknown="error",
        sparse=False,
        dtype=np.dtype(TRANSFORMED_DTYPE),
    )
CATEGORICAL_ENCODER.fit(TRAIN_CAT_MAPPED[CATEGORICAL_MODEL_COLUMNS])
CATEGORICAL_FEATURE_NAMES = CATEGORICAL_ENCODER.get_feature_names_out(
    CATEGORICAL_MODEL_COLUMNS
).tolist()

# Derive exact per-source encoded feature blocks from fitted categories.
CATEGORICAL_GROUP_FEATURE_NAMES = {}
offset = 0
for i, col in enumerate(CATEGORICAL_MODEL_COLUMNS):
    n = len(CATEGORICAL_ENCODER.categories_[i])
    CATEGORICAL_GROUP_FEATURE_NAMES[col] = CATEGORICAL_FEATURE_NAMES[offset:offset+n]
    offset += n
require(offset == len(CATEGORICAL_FEATURE_NAMES), "Categorical feature-name block accounting failed.")

display(pd.DataFrame([
    {
        "feature": c,
        "observed_training_categories": len(support[c]),
        "rare_training_categories": len(CATEGORICAL_POLICY[c]["rare_categories"]),
        "retained_training_categories": len(CATEGORICAL_POLICY[c]["retained_categories"]),
        "encoded_dimensions": len(CATEGORICAL_GROUP_FEATURE_NAMES[c]),
    }
    for c in CATEGORICAL_MODEL_COLUMNS
]))

del TRAIN_CAT_MAPPED
gc.collect()


,feature,observed_training_categories,rare_training_categories,retained_training_categories,encoded_dimensions
0,proto,8,0,8,11
1,state,15,0,15,18
2,service,11,2,9,12


0

In [18]:
# ============================================================
# 21. PERSIST FITTED PREPROCESSING STATE + TRANSPARENT PARAMETERS
# ============================================================
FITTED_PREPROCESSOR_PATH = OUTPUT_DIR / "fitted_preprocessor.joblib"
if FITTED_PREPROCESSOR_PATH.exists():
    hard_fail(f"Refusing to overwrite existing fitted preprocessor: {FITTED_PREPROCESSOR_PATH}")

FITTED_PREPROCESSOR_BUNDLE = {
    "dataset": DATASET_NAME,
    "protocol_version": PROTOCOL_VERSION,
    "preprocessing_policy_hash": PREPROCESSING_POLICY_HASH,
    "fit_partition": "benign_weight_train",
    "raw_model_columns": MODEL_INPUT_COLUMNS,
    "numerical_columns": NUMERICAL_MODEL_COLUMNS,
    "categorical_columns": CATEGORICAL_MODEL_COLUMNS,
    "numerical_imputer": NUMERICAL_IMPUTER,
    "numerical_scaler": NUMERICAL_SCALER,

    "categorical_policy": CATEGORICAL_POLICY,
    "categorical_encoder": CATEGORICAL_ENCODER,
    "categorical_feature_names": CATEGORICAL_FEATURE_NAMES,
    "transformed_dtype": TRANSFORMED_DTYPE,
}
joblib.dump(FITTED_PREPROCESSOR_BUNDLE, FITTED_PREPROCESSOR_PATH, compress=0)
FITTED_PREPROCESSOR_HASH = sha256_file(FITTED_PREPROCESSOR_PATH)
register_artifact(FITTED_PREPROCESSOR_PATH, "Frozen fitted preprocessing bundle", "12")

print("Fitted preprocessor SHA-256:", FITTED_PREPROCESSOR_HASH)


Fitted preprocessor SHA-256: 0088723ea71a531d6cce3fcc753e736eadffd16073cd33068469a01e6890eb86


In [19]:
# ============================================================
# 22–23. FIT-PROVENANCE MANIFEST + POST-FIT LEAKAGE GATE
# ============================================================
fit_meta = pd.read_parquet(
    RAW_METADATA_PATHS["benign_weight_train"],
    columns=["row_id", "partition"],
)
require(set(fit_meta["partition"].astype(str).unique()) == {"benign_weight_train"}, "Fit provenance sidecar contains a non-training partition.")
FIT_ROW_ID_HASH = canonical_string_list_hash(
    fit_meta["row_id"].astype(str).tolist(),
    sort_values=True,
)
require(len(fit_meta) == PARTITION_COUNTS["benign_weight_train"], "Fit provenance row count mismatch.")

FIT_PROVENANCE = {
    "dataset": DATASET_NAME,
    "protocol_version": PROTOCOL_VERSION,
    "fit_partition": "benign_weight_train",
    "fit_row_count": int(len(fit_meta)),
    "canonical_fit_row_id_hash": FIT_ROW_ID_HASH,
    "master_dataset_sha256": MASTER_SHA256,
    "partition_manifest_canonical_hash": PARTITION_MANIFEST_CANONICAL_HASH,
    "raw_feature_policy_hash": RAW_FEATURE_POLICY_HASH,
    "preprocessing_policy_hash": PREPROCESSING_POLICY_HASH,
    "fitted_preprocessor_hash": FITTED_PREPROCESSOR_HASH,
    "fit_source_partitions_used": sorted(FIT_SOURCE_PARTITIONS_USED),
    "leakage_assertions": {
        "benign_early_stop_rows_used_in_fit": 0,
        "benign_threshold_calibration_rows_used_in_fit": 0,
        "development_rows_used_in_fit": 0,
        "final_test_rows_used_in_fit": 0,
        "attack_reserve_unused_rows_used_in_fit": 0,
    },
    "created_at_utc": EXECUTION_UTC,
}
FIT_PROVENANCE_PATH = MANIFEST_DIR / "preprocessing_fit_provenance.json"
write_json_new(FIT_PROVENANCE_PATH, FIT_PROVENANCE)
register_artifact(FIT_PROVENANCE_PATH, "Auditable benign-training-only fit provenance", "12")

require(FIT_SOURCE_PARTITIONS_USED == {"benign_weight_train"}, "A non-training partition entered fitted preprocessing.")
require(all(v == 0 for v in FIT_PROVENANCE["leakage_assertions"].values()), "Post-fit leakage assertion is nonzero.")

LEAKAGE_CHECKLIST_DF = pd.DataFrame([
    {"check": "fit partition is benign_weight_train only", "status": "PASS"},
    {"check": "benign_early_stop rows used in fit", "observed": 0, "status": "PASS"},
    {"check": "benign_threshold_calibration rows used in fit", "observed": 0, "status": "PASS"},
    {"check": "development rows used in fit", "observed": 0, "status": "PASS"},
    {"check": "final_test rows used in fit", "observed": 0, "status": "PASS"},
    {"check": "attack_reserve_unused rows used in fit", "observed": 0, "status": "PASS"},
])
display(LEAKAGE_CHECKLIST_DF)
print("POST-FIT LEAKAGE GATE: PASS")
print("Rows outside benign_weight_train used in fit: 0")
del fit_meta
gc.collect()


,check,status,observed
0,fit partition is benign_weight_train only,PASS,NaN
1,benign_early_stop rows used in fit,PASS,0.0
2,benign_threshold_calibration rows used in fit,PASS,0.0
3,development rows used in fit,PASS,0.0
4,final_test rows used in fit,PASS,0.0
5,attack_reserve_unused rows used in fit,PASS,0.0


POST-FIT LEAKAGE GATE: PASS
Rows outside benign_weight_train used in fit: 0


0

In [20]:
# ============================================================
# 24–27. TRANSFORM ALL FIVE PARTITIONS WITH THE SAME FROZEN FITTED OBJECTS
# ============================================================
TRANSFORMED_FEATURE_NAMES = list(NUMERICAL_MODEL_COLUMNS) + list(CATEGORICAL_FEATURE_NAMES)
FEATURE_ORDER_HASH = canonical_string_list_hash(TRANSFORMED_FEATURE_NAMES, sort_values=False)
TRANSFORMED_NUMERICAL_DIMENSIONS = len(NUMERICAL_MODEL_COLUMNS)
TRANSFORMED_CATEGORICAL_DIMENSIONS = len(TRANSFORMED_FEATURE_NAMES) - TRANSFORMED_NUMERICAL_DIMENSIONS
TOTAL_TRANSFORMED_DIMENSIONS = len(TRANSFORMED_FEATURE_NAMES)

arrow_dtype = pa.float32() if TRANSFORMED_DTYPE == "float32" else pa.float64()
TRANSFORM_SCHEMA = pa.schema([(name, arrow_dtype) for name in TRANSFORMED_FEATURE_NAMES])

TRANSFORMED_PATHS = {p: TRANSFORMED_DIR / f"{p}_transformed.parquet" for p in EXPERIMENTAL_PARTITIONS}
ALIGNMENT_PATHS = {p: TRANSFORMED_DIR / f"{p}_transformed_alignment.parquet" for p in EXPERIMENTAL_PARTITIONS}
TRANSFORM_RESULTS = {}
UNKNOWN_CATEGORY_DIAGNOSTICS = []

for p in EXPERIMENTAL_PARTITIONS:
    target = TRANSFORMED_PATHS[p]
    if target.exists():
        hard_fail(f"Refusing to overwrite transformed artifact: {target}")

    writer = pq.ParquetWriter(target, TRANSFORM_SCHEMA, compression=PARQUET_COMPRESSION_EFFECTIVE)
    rows_written = 0
    nan_count = 0
    posinf_count = 0
    neginf_count = 0
    unknown_accumulator = {c: 0 for c in CATEGORICAL_MODEL_COLUMNS}
    try:
        for frame in iter_parquet_frames(RAW_MODEL_PATHS[p], MODEL_INPUT_COLUMNS, CHUNK_ROWS):
            numeric, _ = normalize_numeric_frame(frame[NUMERICAL_MODEL_COLUMNS])
            X_num = NUMERICAL_IMPUTER.transform(numeric.to_numpy(dtype=np.float64, copy=False))
            X_num = NUMERICAL_SCALER.transform(X_num)
            cat_mapped, unknown_counts = map_categorical_frame(frame[CATEGORICAL_MODEL_COLUMNS], collect_unknown=True)
            X_cat = CATEGORICAL_ENCODER.transform(cat_mapped[CATEGORICAL_MODEL_COLUMNS])
            X = np.concatenate([X_num, X_cat], axis=1).astype(np.dtype(TRANSFORMED_DTYPE), copy=False)
            require(X.shape[1] == TOTAL_TRANSFORMED_DIMENSIONS, f"Transformed dimensionality mismatch in {p}.")
            nan_count += int(np.isnan(X).sum())
            posinf_count += int(np.isposinf(X).sum())
            neginf_count += int(np.isneginf(X).sum())
            for c, n in unknown_counts.items():
                unknown_accumulator[c] += int(n)

            arrays = [pa.array(X[:, i], type=arrow_dtype) for i in range(X.shape[1])]
            writer.write_table(pa.Table.from_arrays(arrays, schema=TRANSFORM_SCHEMA))
            rows_written += len(frame)
            del frame, numeric, X_num, X
            del cat_mapped, X_cat
            gc.collect()
    finally:
        writer.close()

    require(rows_written == PARTITION_COUNTS[p], f"Transformed row count mismatch for {p}.")
    require(nan_count == 0, f"Transformed NaN remains in {p}: {nan_count}")
    require(posinf_count == 0, f"Transformed +Inf remains in {p}: {posinf_count}")
    require(neginf_count == 0, f"Transformed -Inf remains in {p}: {neginf_count}")
    require(parquet_columns(target) == TRANSFORMED_FEATURE_NAMES, f"Transformed feature order mismatch for {p}.")
    _schema = pq.ParquetFile(target).schema_arrow
    require(all(field.type == arrow_dtype for field in _schema),
            f"Transformed dtype mismatch in {p}; expected {TRANSFORMED_DTYPE}.")
    require(parquet_num_rows(target) == rows_written, f"Transformed Parquet metadata row count mismatch for {p}.")

    # Preserve immutable identity in a separate alignment sidecar.
    raw_meta = pd.read_parquet(
        RAW_METADATA_PATHS[p],
        columns=["row_id", "partition", "raw_row_position"],
    )
    require(np.array_equal(raw_meta["raw_row_position"].to_numpy(dtype=np.int64), np.arange(len(raw_meta), dtype=np.int64)), f"Raw alignment position mismatch for {p}.")
    alignment = raw_meta[["row_id", "partition"]].copy()
    alignment["transformed_row_position"] = np.arange(len(alignment), dtype=np.int64)
    write_parquet_new(ALIGNMENT_PATHS[p], alignment, compression=PARQUET_COMPRESSION_EFFECTIVE)
    alignment_hash = canonical_alignment_hash(alignment, "transformed_row_position")
    require(alignment["row_id"].astype("string").reset_index(drop=True).equals(raw_meta["row_id"].astype("string").reset_index(drop=True)), f"Row identity order changed during transformation for {p}.")

    TRANSFORM_RESULTS[p] = {
        "raw_rows": PARTITION_COUNTS[p],
        "transformed_rows": rows_written,
        "nan_count": nan_count,
        "positive_inf_count": posinf_count,
        "negative_inf_count": neginf_count,
        "transformed_artifact_sha256": sha256_file(target),
        "transformed_alignment_sha256": sha256_file(ALIGNMENT_PATHS[p]),
        "transformed_alignment_hash": alignment_hash,
        "raw_alignment_hash": RAW_ALIGNMENT_HASHES[p],
    }
    for c, n in unknown_accumulator.items():
        UNKNOWN_CATEGORY_DIAGNOSTICS.append({
            "partition": p,
            "feature": c,
            "unknown_count": int(n),
        })

    register_artifact(target, f"Frozen transformed model matrix for {p}", "13")
    register_artifact(ALIGNMENT_PATHS[p], f"Transformed row-identity alignment for {p}", "13")
    del raw_meta, alignment
    gc.collect()

TRANSFORM_SCHEMA_DF = pd.DataFrame([
    {
        "partition": p,
        "rows": TRANSFORM_RESULTS[p]["transformed_rows"],
        "dimensions": TOTAL_TRANSFORMED_DIMENSIONS,
        "feature_order_hash": FEATURE_ORDER_HASH,
        "NaN": TRANSFORM_RESULTS[p]["nan_count"],
        "+Inf": TRANSFORM_RESULTS[p]["positive_inf_count"],
        "-Inf": TRANSFORM_RESULTS[p]["negative_inf_count"],
        "schema_status": "PASS",
    }
    for p in EXPERIMENTAL_PARTITIONS
])
display(TRANSFORM_SCHEMA_DF)
ROW_ALIGNMENT_DF = pd.DataFrame([
    {
        "partition": p,
        "raw_rows": PARTITION_COUNTS[p],
        "transformed_rows": TRANSFORM_RESULTS[p]["transformed_rows"],
        "raw_alignment_hash": TRANSFORM_RESULTS[p]["raw_alignment_hash"],
        "transformed_alignment_hash": TRANSFORM_RESULTS[p]["transformed_alignment_hash"],
        "row_identity_order_status": "PASS",
    }
    for p in EXPERIMENTAL_PARTITIONS
])
display(ROW_ALIGNMENT_DF)

if CATEGORICAL_MODEL_COLUMNS:
    UNKNOWN_CATEGORY_DF = pd.DataFrame(UNKNOWN_CATEGORY_DIAGNOSTICS)
    UNKNOWN_CATEGORY_PATH = REPORT_DIR / "unknown_category_transform_summary.csv"
    write_csv_new(UNKNOWN_CATEGORY_PATH, UNKNOWN_CATEGORY_DF)
    register_artifact(UNKNOWN_CATEGORY_PATH, "Mechanical unknown-category transformation counts; not for model decisions", "13")
    _unknown_display = UNKNOWN_CATEGORY_DF.astype("object").copy()
    _ft = _unknown_display["partition"].astype(str).eq("final_test")
    _unknown_display.loc[_ft, "unknown_count"] = "SEALED_IN_ARTIFACT"
    display(_unknown_display)
    print("Unknown-category counts are mechanical compatibility diagnostics only. Final-test counts are stored but sealed from notebook interpretation and MUST NOT drive preprocessing/model redesign.")

print("Final-test status: TRANSFORMED WITH FROZEN BENIGN-TRAINING-FITTED PIPELINE / EVALUATION SEALED")


,partition,rows,dimensions,feature_order_hash,NaN,+Inf,-Inf,schema_status
0,benign_weight_train,1109381,79,5eae35cd2afb6596b2daa463fb535a632b53b51df37153...,0,0,0,PASS
1,benign_early_stop,221878,79,5eae35cd2afb6596b2daa463fb535a632b53b51df37153...,0,0,0,PASS
2,benign_threshold_calibration,332815,79,5eae35cd2afb6596b2daa463fb535a632b53b51df37153...,0,0,0,PASS
3,development,350389,79,5eae35cd2afb6596b2daa463fb535a632b53b51df37153...,0,0,0,PASS
4,final_test,525584,79,5eae35cd2afb6596b2daa463fb535a632b53b51df37153...,0,0,0,PASS


,partition,raw_rows,transformed_rows,raw_alignment_hash,transformed_alignment_hash,row_identity_order_status
0,benign_weight_train,1109381,1109381,9c74f1f67d35ba964bebc3b9f8edc29cfbc74a35f5c41c...,9c74f1f67d35ba964bebc3b9f8edc29cfbc74a35f5c41c...,PASS
1,benign_early_stop,221878,221878,124f81397b61b6bd30f7649dad5660f232f5a882cfe116...,124f81397b61b6bd30f7649dad5660f232f5a882cfe116...,PASS
2,benign_threshold_calibration,332815,332815,16bd8d563e4449538517acba71d7d7c5f15298c565651a...,16bd8d563e4449538517acba71d7d7c5f15298c565651a...,PASS
3,development,350389,350389,e4dfe428873ad4ad9c2b501a14238203ed6f41d8d12689...,e4dfe428873ad4ad9c2b501a14238203ed6f41d8d12689...,PASS
4,final_test,525584,525584,c9f331c05cdbe5244d3c55f8da16ed4fedf57477218b05...,c9f331c05cdbe5244d3c55f8da16ed4fedf57477218b05...,PASS


,partition,feature,unknown_count
0,benign_weight_train,proto,0
1,benign_weight_train,state,0
2,benign_weight_train,service,0
3,benign_early_stop,proto,0
4,benign_early_stop,state,6
5,benign_early_stop,service,0
6,benign_threshold_calibration,proto,1
7,benign_threshold_calibration,state,0
8,benign_threshold_calibration,service,0
9,development,proto,14438


Unknown-category counts are mechanical compatibility diagnostics only. Final-test counts are stored but sealed from notebook interpretation and MUST NOT drive preprocessing/model redesign.
Final-test status: TRANSFORMED WITH FROZEN BENIGN-TRAINING-FITTED PIPELINE / EVALUATION SEALED


In [21]:
# ============================================================
# 28–30. SECTION 14 — DERIVE + VALIDATE + FREEZE DAE CORRUPTION GROUPS
# ============================================================

NUMERICAL_INDICES = list(range(TRANSFORMED_NUMERICAL_DIMENSIONS))
CATEGORICAL_GROUP_INDICES = {}
CATEGORICAL_GROUP_FEATURES = {}
_offset = TRANSFORMED_NUMERICAL_DIMENSIONS
for col in CATEGORICAL_MODEL_COLUMNS:
    names = list(CATEGORICAL_GROUP_FEATURE_NAMES[col])
    idx = list(range(_offset, _offset + len(names)))
    CATEGORICAL_GROUP_INDICES[col] = idx
    CATEGORICAL_GROUP_FEATURES[col] = names
    _offset += len(names)
require(_offset == TOTAL_TRANSFORMED_DIMENSIONS, "UNSW categorical corruption-group indexing does not end at total dimensionality.")

all_indices = list(NUMERICAL_INDICES)
for col in CATEGORICAL_MODEL_COLUMNS:
    all_indices.extend(CATEGORICAL_GROUP_INDICES[col])
require(sorted(all_indices) == list(range(TOTAL_TRANSFORMED_DIMENSIONS)), "UNSW corruption groups do not account for every transformed dimension exactly once.")
require(len(all_indices) == len(set(all_indices)), "UNSW corruption groups overlap.")

CORRUPTION_SUBSTANTIVE = {
    "dataset": DATASET_NAME,
    "protocol_version": PROTOCOL_VERSION,
    "master_dataset_sha256": MASTER_SHA256,
    "partition_manifest_canonical_hash": PARTITION_MANIFEST_CANONICAL_HASH,
    "raw_feature_policy_hash": RAW_FEATURE_POLICY_HASH,
    "preprocessing_policy_hash": PREPROCESSING_POLICY_HASH,
    "fitted_preprocessor_hash": FITTED_PREPROCESSOR_HASH,
    "feature_order_hash": FEATURE_ORDER_HASH,
    "transformed_dtype": TRANSFORMED_DTYPE,
    "total_transformed_dimensions": TOTAL_TRANSFORMED_DIMENSIONS,
    "numerical_indices": NUMERICAL_INDICES,
    "categorical_groups": CATEGORICAL_GROUP_INDICES,
    "categorical_group_feature_names": CATEGORICAL_GROUP_FEATURES,
    "numerical_masking": "searched later",
    "grouped_categorical_masking": "searched later",
    "masking_probabilities_selected_here": False,
    "group_semantics": "proto/state/service are corrupted as complete one-hot source-variable groups, not as independent dummy dimensions",
}

CORRUPTION_GROUP_MANIFEST_HASH = sha256_bytes(canonical_json_bytes(CORRUPTION_SUBSTANTIVE))
CORRUPTION_GROUP_MANIFEST = {
    **CORRUPTION_SUBSTANTIVE,
    "created_at_utc": EXECUTION_UTC,
    "corruption_group_manifest_hash": CORRUPTION_GROUP_MANIFEST_HASH,
}
CORRUPTION_GROUP_MANIFEST_PATH = MANIFEST_DIR / "corruption_group_manifest.json"
write_json_new(CORRUPTION_GROUP_MANIFEST_PATH, CORRUPTION_GROUP_MANIFEST)
register_artifact(CORRUPTION_GROUP_MANIFEST_PATH, "Frozen dataset-specific DAE corruption structure", "14")

corruption_summary = [
    {"group": "numerical", "dimensions": len(NUMERICAL_INDICES)}
]
for k, v in CATEGORICAL_GROUP_INDICES.items():
    corruption_summary.append({"group": k, "dimensions": len(v)})
display(pd.DataFrame(corruption_summary))
print("Corruption-group manifest hash:", CORRUPTION_GROUP_MANIFEST_HASH)


,group,dimensions
0,numerical,38
1,proto,11
2,state,18
3,service,12


Corruption-group manifest hash: 6daf39ed7ef606b16a1bb972289f302647a0defd40c72bfc570e91759e2e19e0


In [22]:
# ============================================================
# 31–32. SECTION 13 — FREEZE TRANSFORMED REPRESENTATION MANIFEST + ARTIFACT HASHES
# ============================================================
PER_PARTITION_REPRESENTATION = {}
for p in EXPERIMENTAL_PARTITIONS:
    PER_PARTITION_REPRESENTATION[p] = {
        "raw_row_count": PARTITION_COUNTS[p],
        "raw_artifact_sha256": RAW_MODEL_HASHES[p],
        "raw_metadata_sha256": RAW_METADATA_HASHES[p],
        "raw_alignment_hash": RAW_ALIGNMENT_HASHES[p],
        "transformed_row_count": TRANSFORM_RESULTS[p]["transformed_rows"],
        "transformed_artifact_sha256": TRANSFORM_RESULTS[p]["transformed_artifact_sha256"],
        "transformed_alignment_sha256": TRANSFORM_RESULTS[p]["transformed_alignment_sha256"],
        "transformed_alignment_hash": TRANSFORM_RESULTS[p]["transformed_alignment_hash"],
        "transformed_dtype": TRANSFORMED_DTYPE,
        "feature_order_hash": FEATURE_ORDER_HASH,
    }

REPRESENTATION_SUBSTANTIVE = {
    "dataset": DATASET_NAME,
    "protocol_version": PROTOCOL_VERSION,
    "master_dataset_sha256": MASTER_SHA256,
    "partition_manifest_canonical_hash": PARTITION_MANIFEST_CANONICAL_HASH,
    "raw_feature_policy_hash": RAW_FEATURE_POLICY_HASH,
    "preprocessing_policy_hash": PREPROCESSING_POLICY_HASH,
    "fitted_preprocessor_hash": FITTED_PREPROCESSOR_HASH,

    "raw_numerical_columns": NUMERICAL_MODEL_COLUMNS,
    "raw_categorical_columns": CATEGORICAL_MODEL_COLUMNS,
    "transformed_numerical_feature_names": NUMERICAL_MODEL_COLUMNS,
    "transformed_numerical_count": TRANSFORMED_NUMERICAL_DIMENSIONS,
    "proto_encoded_feature_names": CATEGORICAL_GROUP_FEATURE_NAMES[CATEGORICAL_MODEL_COLUMNS[0]],
    "proto_encoded_count": len(CATEGORICAL_GROUP_FEATURE_NAMES[CATEGORICAL_MODEL_COLUMNS[0]]),
    "state_encoded_feature_names": CATEGORICAL_GROUP_FEATURE_NAMES[CATEGORICAL_MODEL_COLUMNS[1]],
    "state_encoded_count": len(CATEGORICAL_GROUP_FEATURE_NAMES[CATEGORICAL_MODEL_COLUMNS[1]]),
    "service_encoded_feature_names": CATEGORICAL_GROUP_FEATURE_NAMES[CATEGORICAL_MODEL_COLUMNS[2]],
    "service_encoded_count": len(CATEGORICAL_GROUP_FEATURE_NAMES[CATEGORICAL_MODEL_COLUMNS[2]]),
    "transformed_categorical_dimension_count": TRANSFORMED_CATEGORICAL_DIMENSIONS,
    "categorical_groups": CATEGORICAL_GROUP_INDICES,
    "total_transformed_dimensionality": TOTAL_TRANSFORMED_DIMENSIONS,
    "exact_transformed_feature_order": TRANSFORMED_FEATURE_NAMES,
    "numerical_indices": NUMERICAL_INDICES,
    "feature_order_hash": FEATURE_ORDER_HASH,
    "transformed_dtype": TRANSFORMED_DTYPE,
    "per_partition": PER_PARTITION_REPRESENTATION,
    "corruption_group_manifest_hash": CORRUPTION_GROUP_MANIFEST_HASH,
    "final_test_status": "TRANSFORMED_WITH_FROZEN_BENIGN_TRAINING_FITTED_PIPELINE__EVALUATION_SEALED",
}
TRANSFORMED_REPRESENTATION_MANIFEST_HASH = sha256_bytes(
    canonical_json_bytes(REPRESENTATION_SUBSTANTIVE)
)
TRANSFORMED_REPRESENTATION_MANIFEST = {
    **REPRESENTATION_SUBSTANTIVE,
    "created_at_utc": EXECUTION_UTC,
    "transformed_representation_manifest_hash": TRANSFORMED_REPRESENTATION_MANIFEST_HASH,
}
TRANSFORMED_REPRESENTATION_MANIFEST_PATH = MANIFEST_DIR / "transformed_representation_manifest.json"
write_json_new(TRANSFORMED_REPRESENTATION_MANIFEST_PATH, TRANSFORMED_REPRESENTATION_MANIFEST)
register_artifact(TRANSFORMED_REPRESENTATION_MANIFEST_PATH, "Frozen transformed representation specification", "13")

print("Transformed feature-order hash:", FEATURE_ORDER_HASH)
print("Transformed representation manifest hash:", TRANSFORMED_REPRESENTATION_MANIFEST_HASH)


Transformed feature-order hash: 5eae35cd2afb6596b2daa463fb535a632b53b51df37153eefe40e01750647214
Transformed representation manifest hash: 3a748decdedbb93835bdcda3d50f409385e44b54c0aaaa2f56c6eb2eccb361af


In [23]:
# ============================================================
# 33. FINAL PREPROCESSING / REPRESENTATION INTEGRITY GATE
# ============================================================
# Cross-partition schema, row identity, finiteness, and corruption checks.
for p in EXPERIMENTAL_PARTITIONS:
    require(parquet_columns(TRANSFORMED_PATHS[p]) == TRANSFORMED_FEATURE_NAMES, f"Final schema mismatch for {p}.")
    require(parquet_num_rows(TRANSFORMED_PATHS[p]) == PARTITION_COUNTS[p], f"Final row count mismatch for {p}.")
    _final_schema = pq.ParquetFile(TRANSFORMED_PATHS[p]).schema_arrow
    require(all(field.type == arrow_dtype for field in _final_schema), f"Final transformed dtype mismatch for {p}.")
    require(TRANSFORM_RESULTS[p]["nan_count"] == 0, f"NaN remains in {p}.")
    require(TRANSFORM_RESULTS[p]["positive_inf_count"] == 0, f"+Inf remains in {p}.")
    require(TRANSFORM_RESULTS[p]["negative_inf_count"] == 0, f"-Inf remains in {p}.")
    raw_ids = pd.read_parquet(RAW_METADATA_PATHS[p], columns=["row_id"])["row_id"].astype("string").reset_index(drop=True)
    tx_ids = pd.read_parquet(ALIGNMENT_PATHS[p], columns=["row_id"])["row_id"].astype("string").reset_index(drop=True)
    require(raw_ids.equals(tx_ids), f"Raw/transformed row-id alignment mismatch for {p}.")
    del raw_ids, tx_ids
    gc.collect()

require(not (set(TRANSFORMED_FEATURE_NAMES) & protected_metadata_names), "Protected metadata name entered transformed X.")
require(len(NUMERICAL_INDICES) + sum(len(v) for v in CATEGORICAL_GROUP_INDICES.values()) == TOTAL_TRANSFORMED_DIMENSIONS, "Corruption groups do not cover the transformed representation.")

FINAL_STATUS = "PASS_WITH_NONBLOCKING_WARNINGS" if NONBLOCKING_WARNINGS else "PASS"
NEXT_AUTHORIZED_STAGE = (
    "Exactly one benign-only preformal quantile-transfer smoke AE using the frozen transformed representation; "
    "formal Notebook 01 remains blocked until that smoke gate passes."
)

INTEGRITY_REPORT = {
    "dataset": DATASET_NAME,
    "protocol_version": PROTOCOL_VERSION,
    "notebook_stage": NOTEBOOK_STAGE,
    "created_at_utc": EXECUTION_UTC,
    "protocol_requirements": {
        "fit_partition": "benign_weight_train only",
        "transform_partitions": EXPERIMENTAL_PARTITIONS,
        "final_test_rule": "transformation permitted; influence forbidden",
        "partition_membership_immutable": True,
        "raw_feature_policy_immutable": True,
        "modeling_performed": False,
        "threshold_fitting_performed": False,
    },
    "upstream_verification": {
        "pre00_status": PRE00_READINESS.get("final_status"),
        "pre01_status": PRE01_INTEGRITY.get("status"),
        "pre01_partition_version": PRE01_INTEGRITY.get("partition_version"),
        "pre01_implementation_version": PRE01_INTEGRITY.get("implementation_version"),
        "hashes": UPSTREAM_HASHES,
        "v3_to_pre00_binding": {
            "method": "exact ordered full-dataset (row_id, feature_fingerprint) digest",
            "digest": V3_PRE00_ROW_FP_BINDING_DIGEST,
            "rows": V3_PRE00_ROW_FP_BINDING_ROWS,
            "archived_parent_v2_directory_required": False,
            "status": "PASS",
        },
        "status": "PASS",
    },
    "raw_materialization_results": {
        "partition_counts": PARTITION_COUNTS,
        "raw_model_hashes": RAW_MODEL_HASHES,
        "raw_metadata_hashes": RAW_METADATA_HASHES,
        "status": "PASS",
    },
    "prefit_leakage_gate": {
        "fit_partition": "benign_weight_train",
        "benign_weight_train_attack_rows": 0,
        "status": "PASS",
    },
    "fitted_preprocessing_evidence": {
        "preprocessing_policy_hash": PREPROCESSING_POLICY_HASH,
        "fitted_preprocessor_hash": FITTED_PREPROCESSOR_HASH,
        "numerical_parameter_artifact_sha256": sha256_file(NUMERICAL_PARAMETER_PATH),
        "status": "PASS",
    },
    "postfit_leakage_audit": {
        "fit_row_id_hash": FIT_ROW_ID_HASH,
        "rows_outside_benign_weight_train_used_in_fit": 0,
        "assertions": FIT_PROVENANCE["leakage_assertions"],
        "status": "PASS",
    },
    "transformation_integrity": {
        "results": TRANSFORM_RESULTS,
        "identical_feature_order_hash": FEATURE_ORDER_HASH,
        "transformed_dtype": TRANSFORMED_DTYPE,
        "status": "PASS",
    },
    "representation_freeze": {
        "transformed_representation_manifest_hash": TRANSFORMED_REPRESENTATION_MANIFEST_HASH,
        "feature_order_hash": FEATURE_ORDER_HASH,
        "total_dimensions": TOTAL_TRANSFORMED_DIMENSIONS,
        "status": "PASS",
    },
    "corruption_group_validation": {
        "corruption_group_manifest_hash": CORRUPTION_GROUP_MANIFEST_HASH,
        "coverage_status": "PASS",
        "status": "PASS",
    },
    "upstream_hashes": UPSTREAM_HASHES,
    "preprocessing_policy_hash": PREPROCESSING_POLICY_HASH,
    "fitted_preprocessor_hash": FITTED_PREPROCESSOR_HASH,
    "fit_row_id_hash": FIT_ROW_ID_HASH,
    "feature_order_hash": FEATURE_ORDER_HASH,
    "corruption_group_manifest_hash": CORRUPTION_GROUP_MANIFEST_HASH,
    "transformed_representation_manifest_hash": TRANSFORMED_REPRESENTATION_MANIFEST_HASH,
    "raw_partition_counts": PARTITION_COUNTS,
    "transformation_results": TRANSFORM_RESULTS,
    "rows_outside_benign_weight_train_used_in_fit": 0,
    "final_test_status": "TRANSFORMED WITH FROZEN BENIGN-TRAINING-FITTED PIPELINE / EVALUATION SEALED",
    "nonblocking_warnings": NONBLOCKING_WARNINGS,
    "blocking_failures": BLOCKING_FAILURES,
    "final_status": FINAL_STATUS,
    "next_authorized_stage": NEXT_AUTHORIZED_STAGE,
}

INTEGRITY_JSON_PATH = REPORT_DIR / "preprocessing_integrity_report.json"
INTEGRITY_MD_PATH = REPORT_DIR / "preprocessing_integrity_report.md"
write_json_new(INTEGRITY_JSON_PATH, INTEGRITY_REPORT)

md_lines = [
    f"# PRE-02 Preprocessing / Representation Integrity — {DATASET_NAME}",
    "",
    f"- PRE-00 upstream status: **{PRE00_READINESS.get('final_status')}**",
    f"- PRE-01 upstream status: **{PRE01_INTEGRITY.get('status')}**",
    f"- Fit partition: `benign_weight_train` only",
    f"- Rows outside benign_weight_train used in fit: **0**",
    f"- Total transformed dimensions: **{TOTAL_TRANSFORMED_DIMENSIONS}**",
    f"- Final-test status: **TRANSFORMED WITH FROZEN BENIGN-TRAINING-FITTED PIPELINE / EVALUATION SEALED**",
    f"- PREPROCESSING / REPRESENTATION INTEGRITY: **{FINAL_STATUS}**",
    f"- Next authorized stage: {NEXT_AUTHORIZED_STAGE}",
    "",
    "## Upstream verification",
    "- PRE-00/PRE-01 identities and hashes: PASS",
    "",
    "## Raw materialization",
    "- Five immutable experimental partitions: PASS",
    "",
    "## Pre-fit leakage gate",
    "- benign_weight_train is the only authorized fit partition: PASS",
    "",
    "## Fitted preprocessing evidence",
    f"- Fitted preprocessor SHA-256: `{FITTED_PREPROCESSOR_HASH}`",
    "",
    "## Post-fit leakage audit",
    "- Rows outside benign_weight_train used in fit: 0",
    "",
    "## Transformation integrity",
    f"- Feature-order hash: `{FEATURE_ORDER_HASH}`",
    f"- Transformed dtype: `{TRANSFORMED_DTYPE}`",
    "",
    "## Representation freeze",
    f"- Manifest hash: `{TRANSFORMED_REPRESENTATION_MANIFEST_HASH}`",
    "",
    "## Corruption-group validation",
    f"- Corruption manifest hash: `{CORRUPTION_GROUP_MANIFEST_HASH}`",
    "",
    "## Nonblocking warnings",
]
md_lines.extend([f"- {x}" for x in NONBLOCKING_WARNINGS] or ["- None"])
md_lines.extend(["", "## Blocking failures", "- None"])
write_text_new(INTEGRITY_MD_PATH, "\n".join(md_lines) + "\n")
register_artifact(INTEGRITY_JSON_PATH, "Machine-readable PRE-02 integrity report", "gate")
register_artifact(INTEGRITY_MD_PATH, "Human-readable PRE-02 integrity report", "gate")

FINAL_READINESS_DF = pd.DataFrame([
    {"gate": "PRE-00 readiness", "status": PRE00_READINESS.get("final_status")},
    {"gate": "PRE-01 v3.4 integrity", "status": PRE01_INTEGRITY.get("status")},
    {"gate": "preprocessing fit leakage", "status": "PASS"},
    {"gate": "five-partition transformed schema/finiteness", "status": "PASS"},
    {"gate": "row-identity alignment", "status": "PASS"},
    {"gate": "corruption-group coverage", "status": "PASS"},
    {"gate": "PRE-02 final status", "status": FINAL_STATUS},
])
display(FINAL_READINESS_DF)
print("PREPROCESSING / REPRESENTATION INTEGRITY:", FINAL_STATUS)


,gate,status
0,PRE-00 readiness,PASS_WITH_NONBLOCKING_WARNINGS
1,PRE-01 v3.4 integrity,PASS
2,preprocessing fit leakage,PASS
3,five-partition transformed schema/finiteness,PASS
4,row-identity alignment,PASS
5,corruption-group coverage,PASS
6,PRE-02 final status,PASS_WITH_NONBLOCKING_WARNINGS


PREPROCESSING / REPRESENTATION INTEGRITY: PASS_WITH_NONBLOCKING_WARNINGS


In [24]:
# ============================================================
# 34–36. ARTIFACT INVENTORY + COMPLETION SUMMARY + NEXT-STAGE AUTHORIZATION
# ============================================================
# Register transparent parameter and manifest artifacts that were created earlier.
for path, purpose, section in [
    (PREPROCESSING_POLICY_PATH, "Frozen preprocessing policy", "12"),
    (NUMERICAL_PARAMETER_PATH, "Fitted numerical parameter table", "12"),
    (FITTED_PREPROCESSOR_PATH, "Fitted preprocessing bundle", "12"),
    (FIT_PROVENANCE_PATH, "Fit provenance manifest", "12"),
    (CORRUPTION_GROUP_MANIFEST_PATH, "DAE corruption-group manifest", "14"),
    (TRANSFORMED_REPRESENTATION_MANIFEST_PATH, "Transformed representation manifest", "13"),
]:
    register_artifact(path, purpose, section)

# Deduplicate artifact records by resolved path before freezing inventory.
_inventory_df = pd.DataFrame(ARTIFACT_RECORDS)
if len(_inventory_df):
    _inventory_df = _inventory_df.drop_duplicates(subset=["artifact_path"], keep="last")
    _inventory_df = _inventory_df.sort_values(["section", "artifact_name"], kind="mergesort")
ARTIFACT_INVENTORY_PATH = OUTPUT_DIR / "preprocessing_artifact_inventory.csv"
write_csv_new(ARTIFACT_INVENTORY_PATH, _inventory_df)

raw_counts_text = "\n".join(
    f"    {p}: {PARTITION_COUNTS[p]}" for p in EXPERIMENTAL_PARTITIONS
)
transformed_counts_text = "\n".join(
    f"    {p}: {TRANSFORM_RESULTS[p]['transformed_rows']}" for p in EXPERIMENTAL_PARTITIONS
)

print(f"""
Dataset: {DATASET_NAME}

PRE-00 upstream status: {PRE00_READINESS.get('final_status')}
PRE-01 upstream status: {PRE01_INTEGRITY.get('status')}
PRE-01 partition version: {PRE01_INTEGRITY.get('partition_version')}
PRE-01 implementation: {PRE01_INTEGRITY.get('implementation_version')}

Master SHA-256: {MASTER_SHA256}
Raw feature-policy hash: {RAW_FEATURE_POLICY_HASH}
V3 scientific partition-policy hash: {PARTITION_POLICY_HASH}
Canonical partition-membership hash: {PARTITION_MANIFEST_CANONICAL_HASH}
Direct v3↔PRE-00 row/fingerprint binding digest: {V3_PRE00_ROW_FP_BINDING_DIGEST}
Historical parent-v2 directory required operationally: NO

Fit partition:
benign_weight_train

Raw model feature count: {len(MODEL_INPUT_COLUMNS)}

Numerical raw feature count: {len(NUMERICAL_MODEL_COLUMNS)}
Categorical raw feature count: {len(CATEGORICAL_MODEL_COLUMNS)}

Transformed numerical dimension count: {TRANSFORMED_NUMERICAL_DIMENSIONS}
Transformed categorical dimension count: {TRANSFORMED_CATEGORICAL_DIMENSIONS}
Total transformed dimension count: {TOTAL_TRANSFORMED_DIMENSIONS}

Preprocessing policy hash: {PREPROCESSING_POLICY_HASH}
Fitted preprocessor hash: {FITTED_PREPROCESSOR_HASH}
Transformed feature-order hash: {FEATURE_ORDER_HASH}
Corruption-group manifest hash: {CORRUPTION_GROUP_MANIFEST_HASH}

Raw partition row counts:
{raw_counts_text}

Transformed partition row counts:
{transformed_counts_text}

Rows outside benign_weight_train used in fit: 0

Remaining transformed NaN: 0
Remaining transformed +Inf: 0
Remaining transformed -Inf: 0

Final-test status:
TRANSFORMED WITH FROZEN BENIGN-TRAINING-FITTED PIPELINE
EVALUATION SEALED

PREPROCESSING / REPRESENTATION INTEGRITY:
{FINAL_STATUS}

Next authorized stage:
{NEXT_AUTHORIZED_STAGE}
""")

print("Artifact inventory:", ARTIFACT_INVENTORY_PATH)



Dataset: UNSW-NB15

PRE-00 upstream status: PASS_WITH_NONBLOCKING_WARNINGS
PRE-01 upstream status: PASS
PRE-01 partition version: v3_source_multiplicity_20260919
PRE-01 implementation: v3.4_singleton_tolerance_fix

Master SHA-256: f0447e754fa9ed939ed785e2a8a4d47f3719da2d1fb445134bcb5daf66cf9c54
Raw feature-policy hash: 7defad652ebfaf61febb3167c00f416d399c1aa2f36ee3adf72f21e775f2464b
V3 scientific partition-policy hash: 17122c41630be246c40d9018a97d6f80b58d7422f438673594acb74fd4d2d3fb
Canonical partition-membership hash: eeaff5e0c71fbed867604b56db2de6cea3a4834a78b3dc41ec0e48ae6514c2fd
Direct v3↔PRE-00 row/fingerprint binding digest: e84ab027e5d78f23f1e2ee8b56d5e0c7f178952d3553c74de14df667189eca0d
Historical parent-v2 directory required operationally: NO

Fit partition:
benign_weight_train

Raw model feature count: 41

Numerical raw feature count: 38
Categorical raw feature count: 3

Transformed numerical dimension count: 38
Transformed categorical dimension count: 41
Total transformed d